In [0]:
# ==============================================================================
# Project      : Valorant Champions Tour (VCT) 2025
# Notebook     : Bronze_to_Silver_Matches
# Layer        : Bronze -> Silver
#
#
# Description:
# This notebook reads raw parquet datasets from the Bronze layer,
# performs data cleaning and validation, and writes the cleaned
# datasets into the Silver layer.
# ==============================================================================

# ==============================================================================
# Imports
# ==============================================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import re

# ==============================================================================
# Base Paths
# ==============================================================================

BRONZE_PATH = "/Volumes/workspace/default/matrica/bronze/vct_2025/matches"
SILVER_PATH = "/Volumes/workspace/default/matrica/silver/vct_2025/matches"

print("Project Paths Loaded Successfully")

# ==============================================================================
# Standardize Column Names
#
# Match Type
# ↓
# Match_Type
# ==============================================================================

def standardize_columns(df):

    for column in df.columns:

        new_column = (
            column.strip()
                  .replace(" ", "_")
                  .replace("-", "_")
                  .replace("%", "Percentage")
                  .replace("(", "")
                  .replace(")", "")
                  .replace(",", "")
        )

        # Remove multiple underscores
        new_column = re.sub("_+", "_", new_column)

        # Lower Case
        new_column = (
            column.lower()
                .replace(" ","_")
                .replace("-","_")
        )        
        df = df.withColumnRenamed(column, new_column)

    return df


# ==============================================================================
# Trim String Columns
# ==============================================================================

def trim_string_columns(df):

    string_columns = [

        field.name

        for field in df.schema.fields

        if isinstance(field.dataType, StringType)

    ]

    for column in string_columns:

        df = df.withColumn(column, F.trim(F.col(column)))

    return df


# ==============================================================================
# Convert Blank Strings to NULL
# ==============================================================================

def blank_to_null(df):

    string_columns = [

        field.name

        for field in df.schema.fields

        if isinstance(field.dataType, StringType)

    ]

    for column in string_columns:

        df = df.withColumn(

            column,

            F.when(

                F.trim(F.col(column)) == "",

                None

            ).otherwise(F.col(column))

        )

    return df


# ==============================================================================
# Standardize Text
#
# group stage
# GROUP STAGE
#
# ↓
#
# Group Stage
# ==============================================================================

def standardize_text(df):

    string_columns = [

        field.name

        for field in df.schema.fields

        if isinstance(field.dataType, StringType)

    ]

    for column in string_columns:

        df = df.withColumn(

            column,

            F.initcap(F.col(column))

        )

    return df


# ==============================================================================
# Remove Duplicate Rows
# ==============================================================================

def remove_duplicates(df):

    before = df.count()

    df = df.dropDuplicates()

    after = df.count()

    print(f"Duplicate Rows Removed : {before-after}")

    return df


# ==============================================================================
# Replace NULL in Numeric Columns
# ==============================================================================

def fill_numeric_nulls(df):

    numeric_columns = [

        field.name

        for field in df.schema.fields

        if isinstance(

            field.dataType,

            (

                IntegerType,

                LongType,

                DoubleType,

                FloatType,

                ShortType,

                DecimalType

            )

        )

    ]

    if numeric_columns:

        df = df.fillna(0, subset=numeric_columns)

    return df


# ==============================================================================
# Remove Completely Empty Rows
# ==============================================================================

def remove_empty_rows(df):

    return df.na.drop(how="all")


# ==============================================================================
# Remove Comma from Numeric Strings
#
# 2,900
#
# ↓
#
# 2900
# ==============================================================================

def remove_commas(df, columns):

    for column in columns:

        df = df.withColumn(

            column,

            F.regexp_replace(F.col(column), ",", "")

        )

    return df


# ==============================================================================
# Remove Percentage Symbol
#
# 87%
#
# ↓
#
# 87
# ==============================================================================

def remove_percentage(df, columns):

    for column in columns:

        df = df.withColumn(

            column,

            F.regexp_replace(F.col(column), "%", "")

        )

    return df


# ==============================================================================
# Data Quality Report
# ==============================================================================

def quality_report(df):

    print("="*70)

    print(f"Rows    : {df.count()}")

    print(f"Columns : {len(df.columns)}")

    print("="*70)

    display(

        df.select(

            [

                F.count(

                    F.when(

                        F.col(column).isNull(),

                        column

                    )

                ).alias(column)

                for column in df.columns

            ]

        )

    )


# ==============================================================================
# Write Silver Dataset
# ==============================================================================

def write_silver(df, dataset_name):

    output_path = f"{SILVER_PATH}/{dataset_name}"

    df.write.mode("overwrite").parquet(output_path)

    print(f"Silver Dataset Written Successfully")

    print(output_path)

    # ==============================================================================
# Convert Short Numeric Values
#
# Examples
# 3.3k -> 3300
# 2k   -> 2000
# 1.2m -> 1200000
# ==============================================================================

def convert_k_values(df, column):

    return (
        df
        .withColumn(column, F.lower(F.trim(F.col(column))))
        .withColumn(column, F.regexp_replace(F.col(column), ",", ""))
        .withColumn(
            column,
            F.when(
                F.col(column).rlike(r'^[0-9]*\.?[0-9]+k$'),
                (
                    F.regexp_replace(F.col(column), "k", "").cast("double")
                    * 1000
                ).cast("int")
            )
            .when(
                F.col(column).rlike(r'^[0-9]*\.?[0-9]+m$'),
                (
                    F.regexp_replace(F.col(column), "m", "").cast("double")
                    * 1000000
                ).cast("int")
            )
            .otherwise(F.col(column).cast("int"))
        )
    )


# ==============================================================================
# Validate Silver Dataset
# ==============================================================================

def validate_silver(dataset_name):

    df = spark.read.parquet(f"{SILVER_PATH}/{dataset_name}")

    print("="*70)

    print("Validation Successful")

    print(f"Rows : {df.count()}")

    print(f"Columns : {len(df.columns)}")

    display(df.limit(10))


print("="*70)
print("Common Helper Functions Loaded Successfully")
print("Ready For Bronze → Silver Transformation")
print("="*70)

Project Paths Loaded Successfully
Common Helper Functions Loaded Successfully
Ready For Bronze → Silver Transformation


In [0]:
# ==============================================================================
# Dataset : draft_phase
# Layer   : Bronze -> Silver
# Purpose : Clean Draft Phase Dataset
# ==============================================================================

print("="*80)
print("Processing Dataset : draft_phase")
print("="*80)

# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------

dataset_name = "draft_phase"

df = spark.read.parquet(f"{BRONZE_PATH}/{dataset_name}")

rows_before = df.count()

print(f"Rows Read : {rows_before}")

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report (Before Cleaning)
# ------------------------------------------------------------------------------

print("\nData Quality Report (Before Cleaning)")

quality_report(df)


# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
#
# Match Type
# ↓
# Match_Type
# ------------------------------------------------------------------------------

df = standardize_columns(df)


# ------------------------------------------------------------------------------
# Step 4 : Trim Leading & Trailing Spaces
# ------------------------------------------------------------------------------

df = trim_string_columns(df)


# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------

df = blank_to_null(df)


# ------------------------------------------------------------------------------
# Step 6 : Standardize Text Values
#
# Example:
# group stage → Group Stage
# sentinel → Sentinel
# ------------------------------------------------------------------------------

df = standardize_text(df)


# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------

before_duplicates = df.count()

df = remove_duplicates(df)

after_duplicates = df.count()

duplicates_removed = before_duplicates - after_duplicates


# ------------------------------------------------------------------------------
# Step 8 : Remove Completely Empty Rows
# ------------------------------------------------------------------------------

df = remove_empty_rows(df)


# ------------------------------------------------------------------------------
# Step 9 : Data Quality Report (After Cleaning)
# ------------------------------------------------------------------------------

print("\nData Quality Report (After Cleaning)")

quality_report(df)


# ------------------------------------------------------------------------------
# Step 10 : Write Clean Data to Silver Layer
# ------------------------------------------------------------------------------

write_silver(df, dataset_name)


# ------------------------------------------------------------------------------
# Step 11 : Validate Silver Dataset
# ------------------------------------------------------------------------------

validate_silver(dataset_name)

# another helper function
def log_summary(dataset_name, rows_before, rows_after, duplicates_removed, extra_message=""):

    print("\n" + "=" * 80)
    print("ETL EXECUTION SUMMARY")
    print("=" * 80)
    print(f"Dataset              : {dataset_name}")
    print(f"Rows Read            : {rows_before}")
    print(f"Rows Written         : {rows_after}")
    print(f"Duplicates Removed   : {duplicates_removed}")

    if extra_message:
        print(extra_message)

    print(f"Silver Location      : {SILVER_PATH}/{dataset_name}")
    print("Status               : SUCCESS")
    print("=" * 80)


# ------------------------------------------------------------------------------
# Step 12 : ETL Summary
# ------------------------------------------------------------------------------

print("\n" + "="*80)
print("ETL EXECUTION SUMMARY")
print("="*80)

print(f"Dataset               : {dataset_name}")
print(f"Rows Read             : {rows_before}")
print(f"Rows Written          : {df.count()}")
print(f"Duplicates Removed    : {duplicates_removed}")
print(f"Columns               : {len(df.columns)}")
print(f"Silver Location       : {SILVER_PATH}/{dataset_name}")
print("Status                : SUCCESS")

print("="*80)

Processing Dataset : draft_phase
Rows Read : 2988


Tournament,Stage,Match Type,Match Name,Team,Action,Map
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Xi Lai Gaming,ban,Lotus
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Paper Rex,ban,Abyss
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Xi Lai Gaming,pick,Bind
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Paper Rex,pick,Sunset
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Xi Lai Gaming,ban,Corrode
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Paper Rex,ban,Haven
Valorant Champions 2025,Group Stage,Opening (A),GIANTX vs Sentinels,Sentinels,ban,Ascent
Valorant Champions 2025,Group Stage,Opening (A),GIANTX vs Sentinels,GIANTX,ban,Lotus
Valorant Champions 2025,Group Stage,Opening (A),GIANTX vs Sentinels,Sentinels,pick,Corrode
Valorant Champions 2025,Group Stage,Opening (A),GIANTX vs Sentinels,GIANTX,pick,Sunset



Data Quality Report (Before Cleaning)
Rows    : 2988
Columns : 7


Tournament,Stage,Match Type,Match Name,Team,Action,Map
0,0,0,0,0,0,0


Duplicate Rows Removed : 0

Data Quality Report (After Cleaning)
Rows    : 2988
Columns : 7


tournament,stage,match_type,match_name,team,action,map
0,0,0,0,0,0,0


Silver Dataset Written Successfully
/Volumes/workspace/default/matrica/silver/vct_2025/matches/draft_phase
Validation Successful
Rows : 2988
Columns : 7


tournament,stage,match_type,match_name,team,action,map
Valorant Champions 2025,Group Stage,Opening (d),Dragon Ranger Gaming Vs T1,Dragon Ranger Gaming,Ban,Abyss
Valorant Champions 2025,Group Stage,Winner's (c),Drx Vs Nrg,Mega Minors,Ban,Abyss
Valorant Champions 2025,Group Stage,Winner's (b),Mibr Vs Fnatic,Mibr,Ban,Lotus
Valorant Champions 2025,Group Stage,Decider (d),T1 Vs G2 Esports,T1,Ban,Haven
Valorant Champions 2025,Playoffs,Lower Round 1,Drx Vs G2 Esports,Drx,Ban,Corrode
Valorant Champions 2025,Playoffs,Lower Round 2,Mibr Vs Drx,Mibr,Ban,Haven
Valorant Champions 2025,Playoffs,Upper Final,Fnatic Vs Nrg,Fnatic,Ban,Sunset
Valorant Champions 2025,Playoffs,Lower Final,Fnatic Vs Drx,Drx,Pick,Corrode
Valorant Champions 2025,Playoffs,Lower Final,Fnatic Vs Drx,Fnatic,Pick,Haven
Vct 2025: Americas Stage 2,Group Stage,Week 1,Mibr Vs Krü Esports,Mibr,Ban,Sunset



ETL EXECUTION SUMMARY
Dataset               : draft_phase
Rows Read             : 2988
Rows Written          : 2988
Duplicates Removed    : 0
Columns               : 7
Silver Location       : /Volumes/workspace/default/matrica/silver/vct_2025/matches/draft_phase
Status                : SUCCESS


In [0]:
# ==============================================================================
# Dataset : eco_rounds
# Layer   : Bronze -> Silver
# Purpose : Clean & Transform Economy Rounds Dataset
# ==============================================================================

print("=" * 80)
print("Processing Dataset : eco_rounds")
print("=" * 80)

# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------

dataset_name = "eco_rounds"

df = spark.read.parquet(f"{BRONZE_PATH}/{dataset_name}")

rows_before = df.count()

print(f"Rows Read : {rows_before}")

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report (Before Cleaning)
# ------------------------------------------------------------------------------

print("\nData Quality Report (Before Cleaning)")

quality_report(df)


# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------

df = standardize_columns(df)


# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------

df = trim_string_columns(df)


# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------

df = blank_to_null(df)


# ------------------------------------------------------------------------------
# Step 6 : Standardize Text Columns
# ------------------------------------------------------------------------------

df = standardize_text(df)


# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------

before_duplicates = df.count()

df = remove_duplicates(df)

after_duplicates = df.count()

duplicates_removed = before_duplicates - after_duplicates


# ------------------------------------------------------------------------------
# Step 8 : Clean Loadout_Value
#
# Example:
# 2,900  -> 2900
# ------------------------------------------------------------------------------

df = convert_k_values(df, "Loadout_Value")

# ------------------------------------------------------------------------------
# Step 9 : Clean Remaining_Credits
#
# Example:
# 1,250 -> 1250
# ------------------------------------------------------------------------------
df = convert_k_values(df, "Remaining_Credits")


# ------------------------------------------------------------------------------
# Step 10 : Replace Numeric NULL Values
# ------------------------------------------------------------------------------

df = fill_numeric_nulls(df)


# ------------------------------------------------------------------------------
# Step 11 : Validate Round_Number
#
# Keep only valid rounds (> 0)
# ------------------------------------------------------------------------------

df = df.filter(F.col("Round_Number") > 0)


# ------------------------------------------------------------------------------
# Step 12 : Remove Completely Empty Rows
# ------------------------------------------------------------------------------

df = remove_empty_rows(df)


# ------------------------------------------------------------------------------
# Step 13 : Data Quality Report (After Cleaning)
# ------------------------------------------------------------------------------

print("\nData Quality Report (After Cleaning)")

quality_report(df)

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 14 : Write to Silver Layer
# ------------------------------------------------------------------------------

write_silver(df, dataset_name)


# ------------------------------------------------------------------------------
# Step 15 : Validate Silver Dataset
# ------------------------------------------------------------------------------

validate_silver(dataset_name)


# ------------------------------------------------------------------------------
# Step 16 : ETL Summary
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)

print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_PATH}/{dataset_name}")
print("Status               : SUCCESS")

print("=" * 80)

Processing Dataset : eco_rounds
Rows Read : 25182


Tournament,Stage,Match Type,Match Name,Map,Round Number,Team,Loadout Value,Remaining Credits,Type,Outcome
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Gen.G vs DetonatioN FocusMe,Fracture,8,DetonatioN FocusMe,22.2k,9.4k,Full buy: 20k+,Loss
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Gen.G vs DetonatioN FocusMe,Fracture,9,Gen.G,24.3k,17.5k,Full buy: 20k+,Win
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Gen.G vs DetonatioN FocusMe,Fracture,9,DetonatioN FocusMe,18.7k,2.5k,Semi-buy: 10-20k,Loss
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Gen.G vs DetonatioN FocusMe,Fracture,10,Gen.G,23.3k,18.8k,Full buy: 20k+,Loss
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Gen.G vs DetonatioN FocusMe,Fracture,10,DetonatioN FocusMe,7.0k,9.4k,Semi-eco: 5-10k,Win
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Gen.G vs DetonatioN FocusMe,Fracture,11,Gen.G,23.7k,6.2k,Full buy: 20k+,Win
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Gen.G vs DetonatioN FocusMe,Fracture,11,DetonatioN FocusMe,21.6k,12.6k,Full buy: 20k+,Loss
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Gen.G vs DetonatioN FocusMe,Fracture,12,Gen.G,25.2k,13.7k,Full buy: 20k+,Win
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Gen.G vs DetonatioN FocusMe,Fracture,12,DetonatioN FocusMe,22.0k,4.2k,Full buy: 20k+,Loss
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Gen.G vs DetonatioN FocusMe,Fracture,13,Gen.G,3.6k,0.4k,Eco: 0-5k,Win



Data Quality Report (Before Cleaning)
Rows    : 25182
Columns : 11


Tournament,Stage,Match Type,Match Name,Map,Round Number,Team,Loadout Value,Remaining Credits,Type,Outcome
0,0,0,0,0,0,0,2550,0,0,0


Duplicate Rows Removed : 0

Data Quality Report (After Cleaning)
Rows    : 25182
Columns : 11


tournament,stage,match_type,match_name,map,round_number,team,Loadout_Value,Remaining_Credits,type,outcome
0,0,0,0,0,0,0,0,0,0,0


tournament,stage,match_type,match_name,map,round_number,team,Loadout_Value,Remaining_Credits,type,outcome
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Rex Regum Qeon Vs Zeta Division,Split,14,Zeta Division,3300,7300,Eco: 0-5k,Loss
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Rex Regum Qeon Vs Zeta Division,Haven,1,Rex Regum Qeon,3900,100,Eco: 0-5k,Win
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Rex Regum Qeon Vs Zeta Division,Haven,9,Rex Regum Qeon,10100,7400,Semi-buy: 10-20k,Loss
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Rex Regum Qeon Vs Zeta Division,Haven,11,Rex Regum Qeon,23200,6700,Full Buy: 20k+,Win
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Rex Regum Qeon Vs Zeta Division,Haven,16,Rex Regum Qeon,21500,7000,Full Buy: 20k+,Win
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Paper Rex Vs Boom Esports,Ascent,20,Boom Esports,26200,26100,Full Buy: 20k+,Loss
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Paper Rex Vs Boom Esports,Fracture,5,Boom Esports,22800,12000,Full Buy: 20k+,Loss
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Paper Rex Vs Boom Esports,Fracture,6,Boom Esports,21900,4000,Full Buy: 20k+,Win
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Paper Rex Vs Boom Esports,Fracture,15,Paper Rex,9300,4400,Semi-eco: 5-10k,Loss
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Paper Rex Vs Boom Esports,Fracture,21,Paper Rex,21400,3900,Full Buy: 20k+,Win


Silver Dataset Written Successfully
/Volumes/workspace/default/matrica/silver/vct_2025/matches/eco_rounds
Validation Successful
Rows : 25182
Columns : 11


tournament,stage,match_type,match_name,map,round_number,team,Loadout_Value,Remaining_Credits,type,outcome
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Rex Regum Qeon Vs Zeta Division,Split,14,Zeta Division,3300,7300,Eco: 0-5k,Loss
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Rex Regum Qeon Vs Zeta Division,Haven,1,Rex Regum Qeon,3900,100,Eco: 0-5k,Win
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Rex Regum Qeon Vs Zeta Division,Haven,9,Rex Regum Qeon,10100,7400,Semi-buy: 10-20k,Loss
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Rex Regum Qeon Vs Zeta Division,Haven,11,Rex Regum Qeon,23200,6700,Full Buy: 20k+,Win
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Rex Regum Qeon Vs Zeta Division,Haven,16,Rex Regum Qeon,21500,7000,Full Buy: 20k+,Win
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Paper Rex Vs Boom Esports,Ascent,20,Boom Esports,26200,26100,Full Buy: 20k+,Loss
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Paper Rex Vs Boom Esports,Fracture,5,Boom Esports,22800,12000,Full Buy: 20k+,Loss
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Paper Rex Vs Boom Esports,Fracture,6,Boom Esports,21900,4000,Full Buy: 20k+,Win
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Paper Rex Vs Boom Esports,Fracture,15,Paper Rex,9300,4400,Semi-eco: 5-10k,Loss
Champions Tour 2025: Pacific Stage 1,Group Stage,Week 3,Paper Rex Vs Boom Esports,Fracture,21,Paper Rex,21400,3900,Full Buy: 20k+,Win



ETL EXECUTION SUMMARY
Dataset              : eco_rounds
Rows Read            : 25182
Rows Written         : 25182
Duplicates Removed   : 0
Columns              : 11
Silver Location      : /Volumes/workspace/default/matrica/silver/vct_2025/matches/eco_rounds
Status               : SUCCESS


In [0]:
# ==============================================================================
# Dataset : eco_stats
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate Economy Statistics Dataset
# ==============================================================================

from pyspark.sql import functions as F

print("=" * 80)
print("Processing Dataset : eco_stats")
print("=" * 80)

# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------

dataset_name = "eco_stats"

df = spark.read.parquet(f"{BRONZE_PATH}/{dataset_name}")

rows_before = df.count()

print(f"Rows Read : {rows_before}")

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report (Before Cleaning)
# ------------------------------------------------------------------------------

print("\nData Quality Report (Before Cleaning)")

quality_report(df)


# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------

df = standardize_columns(df)


# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------

df = trim_string_columns(df)


# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------

df = blank_to_null(df)


# ------------------------------------------------------------------------------
# Step 6 : Standardize Text Columns
# ------------------------------------------------------------------------------

df = standardize_text(df)


# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------

before_duplicates = df.count()

df = remove_duplicates(df)

after_duplicates = df.count()

duplicates_removed = before_duplicates - after_duplicates


# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------

df = fill_numeric_nulls(df)


# ------------------------------------------------------------------------------
# Step 9 : Business Rule Validation
#
# Won should never be greater than Initiated
# ------------------------------------------------------------------------------

invalid_records = df.filter(F.col("Won") > F.col("Initiated")).count()

print(f"Invalid Records (Won > Initiated) : {invalid_records}")

# Keep only valid records
df = df.filter(F.col("Won") <= F.col("Initiated"))


# ------------------------------------------------------------------------------
# Step 10 : Remove Completely Empty Rows
# ------------------------------------------------------------------------------

df = remove_empty_rows(df)


# ------------------------------------------------------------------------------
# Step 11 : Data Quality Report (After Cleaning)
# ------------------------------------------------------------------------------

print("\nData Quality Report (After Cleaning)")

quality_report(df)

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 12 : Write to Silver Layer
# ------------------------------------------------------------------------------

write_silver(df, dataset_name)


# ------------------------------------------------------------------------------
# Step 13 : Validate Silver Dataset
# ------------------------------------------------------------------------------

validate_silver(dataset_name)


# ------------------------------------------------------------------------------
# Step 14 : ETL Summary
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)

print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Business Rule Errors : {invalid_records}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_PATH}/{dataset_name}")
print("Status               : SUCCESS")

print("=" * 80)

Processing Dataset : eco_stats
Rows Read : 13940


Tournament,Stage,Match Type,Match Name,Map,Team,Type,Initiated,Won
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,Paper Rex,Pistol Won,null,1
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,Paper Rex,Eco (won),4,2
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,Paper Rex,$ (won),0,0
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,Paper Rex,$$ (won),3,2
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,Paper Rex,$$$ (won),15,9
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,Xi Lai Gaming,Pistol Won,null,1
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,Xi Lai Gaming,Eco (won),3,1
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,Xi Lai Gaming,$ (won),1,0
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,Xi Lai Gaming,$$ (won),8,3
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,Xi Lai Gaming,$$$ (won),10,5



Data Quality Report (Before Cleaning)
Rows    : 13940
Columns : 9


Tournament,Stage,Match Type,Match Name,Map,Team,Type,Initiated,Won
0,0,0,0,0,0,0,2788,0


Duplicate Rows Removed : 0
Invalid Records (Won > Initiated) : 2249

Data Quality Report (After Cleaning)
Rows    : 11691
Columns : 9


tournament,stage,match_type,match_name,map,team,type,initiated,won
0,0,0,0,0,0,0,0,0


tournament,stage,match_type,match_name,map,team,type,initiated,won
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Sunset,Paper Rex,$$$ (won),10,7
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Sunset,Xi Lai Gaming,Pistol Won,0,0
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Haven,Sentinels,$$ (won),6,3
Valorant Champions 2025,Group Stage,Opening (c),Nrg Vs Edward Gaming,Abyss,Edward Gaming,$$ (won),5,2
Valorant Champions 2025,Group Stage,Opening (c),Nrg Vs Edward Gaming,All Maps,Edward Gaming,$ (won),2,0
Valorant Champions 2025,Group Stage,Opening (c),Team Liquid Vs Drx,Abyss,Team Liquid,Pistol Won,0,0
Valorant Champions 2025,Group Stage,Opening (d),Dragon Ranger Gaming Vs T1,Sunset,Dragon Ranger Gaming,$ (won),1,1
Valorant Champions 2025,Group Stage,Opening (d),Dragon Ranger Gaming Vs T1,Sunset,T1,$$$ (won),19,9
Valorant Champions 2025,Group Stage,Opening (d),Dragon Ranger Gaming Vs T1,All Maps,T1,$$ (won),8,4
Valorant Champions 2025,Group Stage,Opening (d),G2 Esports Vs Team Heretics,Corrode,G2 Esports,$$$ (won),15,9


Silver Dataset Written Successfully
/Volumes/workspace/default/matrica/silver/vct_2025/matches/eco_stats
Validation Successful
Rows : 11691
Columns : 9


tournament,stage,match_type,match_name,map,team,type,initiated,won
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Sunset,Paper Rex,$$$ (won),10,7
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Sunset,Xi Lai Gaming,Pistol Won,0,0
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Haven,Sentinels,$$ (won),6,3
Valorant Champions 2025,Group Stage,Opening (c),Nrg Vs Edward Gaming,Abyss,Edward Gaming,$$ (won),5,2
Valorant Champions 2025,Group Stage,Opening (c),Nrg Vs Edward Gaming,All Maps,Edward Gaming,$ (won),2,0
Valorant Champions 2025,Group Stage,Opening (c),Team Liquid Vs Drx,Abyss,Team Liquid,Pistol Won,0,0
Valorant Champions 2025,Group Stage,Opening (d),Dragon Ranger Gaming Vs T1,Sunset,Dragon Ranger Gaming,$ (won),1,1
Valorant Champions 2025,Group Stage,Opening (d),Dragon Ranger Gaming Vs T1,Sunset,T1,$$$ (won),19,9
Valorant Champions 2025,Group Stage,Opening (d),Dragon Ranger Gaming Vs T1,All Maps,T1,$$ (won),8,4
Valorant Champions 2025,Group Stage,Opening (d),G2 Esports Vs Team Heretics,Corrode,G2 Esports,$$$ (won),15,9



ETL EXECUTION SUMMARY
Dataset              : eco_stats
Rows Read            : 13940
Rows Written         : 11691
Duplicates Removed   : 0
Business Rule Errors : 2249
Columns              : 9
Silver Location      : /Volumes/workspace/default/matrica/silver/vct_2025/matches/eco_stats
Status               : SUCCESS


In [0]:
# ==============================================================================
# Dataset : kills
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate Kills Dataset
# ==============================================================================

from pyspark.sql import functions as F

print("=" * 80)
print("Processing Dataset : kills")
print("=" * 80)

# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------

dataset_name = "kills"

df = spark.read.parquet(f"{BRONZE_PATH}/{dataset_name}")

rows_before = df.count()

print(f"Rows Read : {rows_before}")

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (Before Cleaning)")

quality_report(df)


# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------

df = standardize_columns(df)


# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------

df = trim_string_columns(df)


# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------

df = blank_to_null(df)


# ------------------------------------------------------------------------------
# Step 6 : Standardize Text Values
# ------------------------------------------------------------------------------

df = standardize_text(df)


# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------

before_duplicates = df.count()

df = remove_duplicates(df)

after_duplicates = df.count()

duplicates_removed = before_duplicates - after_duplicates


# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------

df = fill_numeric_nulls(df)


# ------------------------------------------------------------------------------
# Step 9 : Remove Invalid Kill Counts
#
# Kill values cannot be negative
# ------------------------------------------------------------------------------

invalid_kills = df.filter(
    (F.col("Player_Kills") < 0) |
    (F.col("Enemy_Kills") < 0)
).count()

print(f"Invalid Kill Records : {invalid_kills}")

df = df.filter(
    (F.col("Player_Kills") >= 0) &
    (F.col("Enemy_Kills") >= 0)
)


# ------------------------------------------------------------------------------
# Step 10 : Recalculate Difference
#
# Difference = Player_Kills - Enemy_Kills
# ------------------------------------------------------------------------------

df = df.withColumn(
    "Difference",
    F.col("Player_Kills") - F.col("Enemy_Kills")
)


# ------------------------------------------------------------------------------
# Step 11 : Remove Empty Rows
# ------------------------------------------------------------------------------

df = remove_empty_rows(df)


# ------------------------------------------------------------------------------
# Step 12 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (After Cleaning)")

quality_report(df)

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 13 : Write Silver Dataset
# ------------------------------------------------------------------------------

write_silver(df, dataset_name)


# ------------------------------------------------------------------------------
# Step 14 : Validate Silver Dataset
# ------------------------------------------------------------------------------

validate_silver(dataset_name)


# ------------------------------------------------------------------------------
# Step 15 : ETL Summary
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)

print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Invalid Kill Records : {invalid_kills}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_PATH}/{dataset_name}")
print("Status               : SUCCESS")

print("=" * 80)

Processing Dataset : kills
Rows Read : 104325


Tournament,Stage,Match Type,Match Name,Map,Player Team,Player,Enemy Team,Enemy,Player Kills,Enemy Kills,Difference,Kill Type
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Paper Rex,PatMen,Xi Lai Gaming,NoMan,4,6,-2,All Kills
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Paper Rex,PatMen,Xi Lai Gaming,happywei,3,7,-4,All Kills
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Paper Rex,PatMen,Xi Lai Gaming,Viva,3,2,1,All Kills
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Paper Rex,PatMen,Xi Lai Gaming,coconut,5,5,0,All Kills
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Paper Rex,PatMen,Xi Lai Gaming,Rarga,3,6,-3,All Kills
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Paper Rex,d4v41,Xi Lai Gaming,NoMan,5,9,-4,All Kills
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Paper Rex,d4v41,Xi Lai Gaming,happywei,6,2,4,All Kills
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Paper Rex,d4v41,Xi Lai Gaming,Viva,5,4,1,All Kills
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Paper Rex,d4v41,Xi Lai Gaming,coconut,5,4,1,All Kills
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Paper Rex,d4v41,Xi Lai Gaming,Rarga,4,6,-2,All Kills



Data Quality Report (Before Cleaning)
Rows    : 104325
Columns : 13


Tournament,Stage,Match Type,Match Name,Map,Player Team,Player,Enemy Team,Enemy,Player Kills,Enemy Kills,Difference,Kill Type
0,0,0,0,0,0,0,0,0,43364,43364,43460,0


Duplicate Rows Removed : 0
Invalid Kill Records : 0

Data Quality Report (After Cleaning)
Rows    : 104325
Columns : 13


tournament,stage,match_type,match_name,map,player_team,player,enemy_team,enemy,player_kills,enemy_kills,Difference,kill_type
0,0,0,0,0,0,0,0,0,0,0,0,0


tournament,stage,match_type,match_name,map,player_team,player,enemy_team,enemy,player_kills,enemy_kills,Difference,kill_type
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,All Maps,Paper Rex,Patmen,Xi Lai Gaming,Happywei,3,7,-4,All Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,All Maps,Paper Rex,D4v41,Xi Lai Gaming,Noman,0,1,-1,First Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,All Maps,Paper Rex,D4v41,Xi Lai Gaming,Rarga,0,0,0,First Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,All Maps,Paper Rex,Something,Xi Lai Gaming,Noman,2,0,2,First Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,All Maps,Paper Rex,Patmen,Xi Lai Gaming,Rarga,0,0,0,Op Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,All Maps,Paper Rex,Something,Xi Lai Gaming,Noman,2,0,2,Op Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,Paper Rex,Patmen,Xi Lai Gaming,Viva,2,2,0,All Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,Paper Rex,D4v41,Xi Lai Gaming,Noman,1,5,-4,All Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,Paper Rex,F0rsaken,Xi Lai Gaming,Happywei,5,3,2,All Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,Paper Rex,Jinggg,Xi Lai Gaming,Noman,7,4,3,All Kills


Silver Dataset Written Successfully
/Volumes/workspace/default/matrica/silver/vct_2025/matches/kills
Validation Successful
Rows : 104325
Columns : 13


tournament,stage,match_type,match_name,map,player_team,player,enemy_team,enemy,player_kills,enemy_kills,Difference,kill_type
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,All Maps,Paper Rex,Patmen,Xi Lai Gaming,Happywei,3,7,-4,All Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,All Maps,Paper Rex,D4v41,Xi Lai Gaming,Noman,0,1,-1,First Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,All Maps,Paper Rex,D4v41,Xi Lai Gaming,Rarga,0,0,0,First Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,All Maps,Paper Rex,Something,Xi Lai Gaming,Noman,2,0,2,First Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,All Maps,Paper Rex,Patmen,Xi Lai Gaming,Rarga,0,0,0,Op Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,All Maps,Paper Rex,Something,Xi Lai Gaming,Noman,2,0,2,Op Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,Paper Rex,Patmen,Xi Lai Gaming,Viva,2,2,0,All Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,Paper Rex,D4v41,Xi Lai Gaming,Noman,1,5,-4,All Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,Paper Rex,F0rsaken,Xi Lai Gaming,Happywei,5,3,2,All Kills
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,Paper Rex,Jinggg,Xi Lai Gaming,Noman,7,4,3,All Kills



ETL EXECUTION SUMMARY
Dataset              : kills
Rows Read            : 104325
Rows Written         : 104325
Duplicates Removed   : 0
Invalid Kill Records : 0
Columns              : 13
Silver Location      : /Volumes/workspace/default/matrica/silver/vct_2025/matches/kills
Status               : SUCCESS


In [0]:
# ==============================================================================
# Dataset : kills_stats
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate Player Kill Statistics
# ==============================================================================

from pyspark.sql import functions as F

print("=" * 80)
print("Processing Dataset : kills_stats")
print("=" * 80)

# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------

dataset_name = "kills_stats"

df = spark.read.parquet(f"{BRONZE_PATH}/{dataset_name}")

rows_before = df.count()

print(f"Rows Read : {rows_before}")

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (Before Cleaning)")

quality_report(df)


# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------

df = standardize_columns(df)


# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------

df = trim_string_columns(df)


# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------

df = blank_to_null(df)


# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------

df = standardize_text(df)


# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------

before_duplicates = df.count()

df = remove_duplicates(df)

after_duplicates = df.count()

duplicates_removed = before_duplicates - after_duplicates


# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------

df = fill_numeric_nulls(df)


# ------------------------------------------------------------------------------
# Step 9 : Validate Numeric Columns
#
# These values should never be negative.
# ------------------------------------------------------------------------------

numeric_columns = [
    "2k",
    "3k",
    "4k",
    "5k",
    "1v1",
    "1v2",
    "1v3",
    "1v4",
    "1v5",
    "Econ",
    "Spike_Plants",
    "Spike_Defuses"
]

invalid_records = None

for column in numeric_columns:

    condition = F.col(f"`{column}`") < 0

    if invalid_records is None:
        invalid_records = condition
    else:
        invalid_records = invalid_records | condition


invalid_count = df.filter(invalid_records).count()

print(f"Invalid Numeric Records : {invalid_count}")

df = df.filter(~invalid_records)


# ------------------------------------------------------------------------------
# Step 10 : Convert 1v5 to Integer
#
# (Only if all values are whole numbers)
# ------------------------------------------------------------------------------

df = df.withColumn("1v5", F.round(F.col("1v5"), 0).cast("int"))


# ------------------------------------------------------------------------------
# Step 11 : Remove Empty Rows
# ------------------------------------------------------------------------------

df = remove_empty_rows(df)


# ------------------------------------------------------------------------------
# Step 12 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (After Cleaning)")

quality_report(df)

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 13 : Write Silver Dataset
# ------------------------------------------------------------------------------

write_silver(df, dataset_name)


# ------------------------------------------------------------------------------
# Step 14 : Validate Silver Dataset
# ------------------------------------------------------------------------------

validate_silver(dataset_name)


# ------------------------------------------------------------------------------
# Step 15 : ETL Summary
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)

print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Invalid Records      : {invalid_count}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_PATH}/{dataset_name}")
print("Status               : SUCCESS")

print("=" * 80)

Processing Dataset : kills_stats
Rows Read : 13912


Tournament,Stage,Match Type,Match Name,Map,Team,Player,Agents,2k,3k,4k,5k,1v1,1v2,1v3,1v4,1v5,Econ,Spike Plants,Spike Defuses
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Paper Rex,Jinggg,raze,8,5,null,null,null,null,null,null,null,73,0,2
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Paper Rex,f0rsakeN,"brimstone, omen",5,1,3,null,null,1,null,null,null,66,1,0
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Paper Rex,d4v41,"sage, viper",2,2,null,null,3,null,null,null,null,43,5,3
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Paper Rex,PatMen,"fade, viper",4,null,null,null,null,null,null,null,null,36,4,0
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Paper Rex,something,"sova, yoru",7,1,1,1,1,1,null,null,null,78,1,2
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Xi Lai Gaming,NoMan,"neon, skye",10,1,null,null,null,null,null,null,null,67,1,0
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Xi Lai Gaming,coconut,"gekko, omen",7,null,null,null,null,null,null,null,null,38,9,0
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Xi Lai Gaming,Rarga,"raze, yoru",6,1,null,null,null,1,null,null,null,48,0,1
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Xi Lai Gaming,Viva,"brimstone, sova",2,null,1,null,1,1,null,1,null,39,6,0
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,All Maps,Xi Lai Gaming,happywei,"cypher, viper",1,2,null,null,null,null,null,null,null,53,1,1



Data Quality Report (Before Cleaning)
Rows    : 13912
Columns : 20


Tournament,Stage,Match Type,Match Name,Map,Team,Player,Agents,2k,3k,4k,5k,1v1,1v2,1v3,1v4,1v5,Econ,Spike Plants,Spike Defuses
0,0,0,0,0,0,0,0,820,5210,11077,13504,10485,11984,13368,13810,13902,0,0,0


Duplicate Rows Removed : 0
Invalid Numeric Records : 0

Data Quality Report (After Cleaning)
Rows    : 13912
Columns : 20


tournament,stage,match_type,match_name,map,team,player,agents,2k,3k,4k,5k,1v1,1v2,1v3,1v4,1v5,econ,spike_plants,spike_defuses
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


tournament,stage,match_type,match_name,map,team,player,agents,2k,3k,4k,5k,1v1,1v2,1v3,1v4,1v5,econ,spike_plants,spike_defuses
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,All Maps,Sentinels,N4rrate,"Fade, Sova",8,2,1,0,2,0,0,0,0,60,7,2
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Corrode,Sentinels,Johnqt,Viper,2,1,0,0,0,0,0,0,0,74,2,0
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Corrode,Sentinels,Bang,Omen,3,0,0,0,0,0,0,0,0,45,3,0
Valorant Champions 2025,Group Stage,Opening (c),Nrg Vs Edward Gaming,Abyss,Mega Minors,Ethan,Omen,4,1,0,0,0,0,0,0,0,49,2,0
Valorant Champions 2025,Group Stage,Opening (c),Nrg Vs Edward Gaming,Abyss,Edward Gaming,Zmjjkk,Yoru,4,2,0,0,0,0,0,0,0,44,1,0
Valorant Champions 2025,Group Stage,Opening (c),Team Liquid Vs Drx,Abyss,Drx,Free1ng,Cypher,2,0,0,0,0,0,0,0,0,44,0,0
Valorant Champions 2025,Group Stage,Opening (d),Dragon Ranger Gaming Vs T1,Sunset,Dragon Ranger Gaming,Flex1n,Omen,3,1,0,0,0,0,0,0,0,47,4,0
Valorant Champions 2025,Group Stage,Opening (b),Bilibili Gaming Vs Mibr,All Maps,Mibr,Cortezia,"Killjoy, Viper",5,2,0,0,1,0,0,0,0,62,3,1
Valorant Champions 2025,Group Stage,Opening (b),Bilibili Gaming Vs Mibr,All Maps,Mibr,Aspas,"Neon, Waylay",6,1,1,1,0,0,0,0,0,72,0,1
Valorant Champions 2025,Group Stage,Opening (b),Bilibili Gaming Vs Mibr,Haven,Bilibili Gaming,Levius,Cypher,2,0,0,0,0,1,0,0,0,19,0,1


Silver Dataset Written Successfully
/Volumes/workspace/default/matrica/silver/vct_2025/matches/kills_stats
Validation Successful
Rows : 13912
Columns : 20


tournament,stage,match_type,match_name,map,team,player,agents,2k,3k,4k,5k,1v1,1v2,1v3,1v4,1v5,econ,spike_plants,spike_defuses
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,All Maps,Sentinels,N4rrate,"Fade, Sova",8,2,1,0,2,0,0,0,0,60,7,2
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Corrode,Sentinels,Johnqt,Viper,2,1,0,0,0,0,0,0,0,74,2,0
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Corrode,Sentinels,Bang,Omen,3,0,0,0,0,0,0,0,0,45,3,0
Valorant Champions 2025,Group Stage,Opening (c),Nrg Vs Edward Gaming,Abyss,Mega Minors,Ethan,Omen,4,1,0,0,0,0,0,0,0,49,2,0
Valorant Champions 2025,Group Stage,Opening (c),Nrg Vs Edward Gaming,Abyss,Edward Gaming,Zmjjkk,Yoru,4,2,0,0,0,0,0,0,0,44,1,0
Valorant Champions 2025,Group Stage,Opening (c),Team Liquid Vs Drx,Abyss,Drx,Free1ng,Cypher,2,0,0,0,0,0,0,0,0,44,0,0
Valorant Champions 2025,Group Stage,Opening (d),Dragon Ranger Gaming Vs T1,Sunset,Dragon Ranger Gaming,Flex1n,Omen,3,1,0,0,0,0,0,0,0,47,4,0
Valorant Champions 2025,Group Stage,Opening (b),Bilibili Gaming Vs Mibr,All Maps,Mibr,Cortezia,"Killjoy, Viper",5,2,0,0,1,0,0,0,0,62,3,1
Valorant Champions 2025,Group Stage,Opening (b),Bilibili Gaming Vs Mibr,All Maps,Mibr,Aspas,"Neon, Waylay",6,1,1,1,0,0,0,0,0,72,0,1
Valorant Champions 2025,Group Stage,Opening (b),Bilibili Gaming Vs Mibr,Haven,Bilibili Gaming,Levius,Cypher,2,0,0,0,0,1,0,0,0,19,0,1



ETL EXECUTION SUMMARY
Dataset              : kills_stats
Rows Read            : 13912
Rows Written         : 13912
Duplicates Removed   : 0
Invalid Records      : 0
Columns              : 20
Silver Location      : /Volumes/workspace/default/matrica/silver/vct_2025/matches/kills_stats
Status               : SUCCESS


In [0]:
# ==============================================================================
# Dataset : maps_played
# Layer   : Bronze -> Silver
# Purpose : Clean & Standardize Maps Played Dataset
# ==============================================================================

from pyspark.sql import functions as F

print("=" * 80)
print("Processing Dataset : maps_played")
print("=" * 80)

# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------

dataset_name = "maps_played"

df = spark.read.parquet(f"{BRONZE_PATH}/{dataset_name}")

rows_before = df.count()

print(f"Rows Read : {rows_before}")

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (Before Cleaning)")

quality_report(df)


# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------

df = standardize_columns(df)


# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------

df = trim_string_columns(df)


# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------

df = blank_to_null(df)


# ------------------------------------------------------------------------------
# Step 6 : Standardize Text Values
# ------------------------------------------------------------------------------

df = standardize_text(df)


# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------

before_duplicates = df.count()

df = remove_duplicates(df)

after_duplicates = df.count()

duplicates_removed = before_duplicates - after_duplicates


# ------------------------------------------------------------------------------
# Step 8 : Remove Records with Missing Map
# ------------------------------------------------------------------------------

missing_maps = df.filter(F.col("Map").isNull()).count()

print(f"Rows Removed (Missing Map): {missing_maps}")

df = df.filter(F.col("Map").isNotNull())


# ------------------------------------------------------------------------------
# Step 9 : Remove Completely Empty Rows
# ------------------------------------------------------------------------------

df = remove_empty_rows(df)


# ------------------------------------------------------------------------------
# Step 10 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (After Cleaning)")

quality_report(df)

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 11 : Write Silver Dataset
# ------------------------------------------------------------------------------

write_silver(df, dataset_name)


# ------------------------------------------------------------------------------
# Step 12 : Validate Silver Dataset
# ------------------------------------------------------------------------------

validate_silver(dataset_name)


# ------------------------------------------------------------------------------
# Step 13 : ETL Summary
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)

print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Missing Maps Removed : {missing_maps}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_PATH}/{dataset_name}")
print("Status               : SUCCESS")

print("=" * 80)

Processing Dataset : maps_played
Rows Read : 1277


Tournament,Stage,Match Type,Match Name,Map
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Sunset
Valorant Champions 2025,Group Stage,Opening (A),GIANTX vs Sentinels,Corrode
Valorant Champions 2025,Group Stage,Opening (A),GIANTX vs Sentinels,Sunset
Valorant Champions 2025,Group Stage,Opening (A),GIANTX vs Sentinels,Haven
Valorant Champions 2025,Group Stage,Opening (C),NRG vs EDward Gaming,Abyss
Valorant Champions 2025,Group Stage,Opening (C),NRG vs EDward Gaming,Corrode
Valorant Champions 2025,Group Stage,Opening (C),Team Liquid vs DRX,Abyss
Valorant Champions 2025,Group Stage,Opening (C),Team Liquid vs DRX,Bind
Valorant Champions 2025,Group Stage,Opening (D),Dragon Ranger Gaming vs T1,Lotus



Data Quality Report (Before Cleaning)
Rows    : 1277
Columns : 5


Tournament,Stage,Match Type,Match Name,Map
0,0,0,0,0


Duplicate Rows Removed : 0
Rows Removed (Missing Map): 0

Data Quality Report (After Cleaning)
Rows    : 1277
Columns : 5


tournament,stage,match_type,match_name,map
0,0,0,0,0


tournament,stage,match_type,match_name,map
Valorant Champions 2025,Group Stage,Opening (b),Rex Regum Qeon Vs Fnatic,Ascent
Valorant Champions 2025,Group Stage,Winner's (a),Paper Rex Vs Giantx,Ascent
Valorant Champions 2025,Group Stage,Winner's (b),Mibr Vs Fnatic,Bind
Valorant Champions 2025,Group Stage,Decider (d),T1 Vs G2 Esports,Corrode
Valorant Champions 2025,Playoffs,Upper Quarterfinals,Fnatic Vs Drx,Haven
Valorant Champions 2025,Playoffs,Upper Quarterfinals,Nrg Vs Giantx,Lotus
Valorant Champions 2025,Playoffs,Lower Round 2,Paper Rex Vs Team Heretics,Bind
Valorant Champions 2025,Playoffs,Lower Round 2,Mibr Vs Drx,Bind
Vct 2025: Americas Stage 2,Group Stage,Week 1,Mibr Vs Krü Esports,Haven
Vct 2025: Americas Stage 2,Group Stage,Week 3,Sentinels Vs Cloud9,Haven


Silver Dataset Written Successfully
/Volumes/workspace/default/matrica/silver/vct_2025/matches/maps_played
Validation Successful
Rows : 1277
Columns : 5


tournament,stage,match_type,match_name,map
Valorant Champions 2025,Group Stage,Opening (b),Rex Regum Qeon Vs Fnatic,Ascent
Valorant Champions 2025,Group Stage,Winner's (a),Paper Rex Vs Giantx,Ascent
Valorant Champions 2025,Group Stage,Winner's (b),Mibr Vs Fnatic,Bind
Valorant Champions 2025,Group Stage,Decider (d),T1 Vs G2 Esports,Corrode
Valorant Champions 2025,Playoffs,Upper Quarterfinals,Fnatic Vs Drx,Haven
Valorant Champions 2025,Playoffs,Upper Quarterfinals,Nrg Vs Giantx,Lotus
Valorant Champions 2025,Playoffs,Lower Round 2,Paper Rex Vs Team Heretics,Bind
Valorant Champions 2025,Playoffs,Lower Round 2,Mibr Vs Drx,Bind
Vct 2025: Americas Stage 2,Group Stage,Week 1,Mibr Vs Krü Esports,Haven
Vct 2025: Americas Stage 2,Group Stage,Week 3,Sentinels Vs Cloud9,Haven



ETL EXECUTION SUMMARY
Dataset              : maps_played
Rows Read            : 1277
Rows Written         : 1277
Duplicates Removed   : 0
Missing Maps Removed : 0
Columns              : 5
Silver Location      : /Volumes/workspace/default/matrica/silver/vct_2025/matches/maps_played
Status               : SUCCESS


In [0]:
# ==============================================================================
# Dataset : maps_scores
# Layer   : Bronze -> Silver
# Purpose : Clean, Validate & Transform Map Scores Dataset
# ==============================================================================

from pyspark.sql import functions as F

print("=" * 80)
print("Processing Dataset : maps_scores")
print("=" * 80)

# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------

dataset_name = "maps_scores"

df = spark.read.parquet(f"{BRONZE_PATH}/{dataset_name}")

rows_before = df.count()

print(f"Rows Read : {rows_before}")

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (Before Cleaning)")

quality_report(df)


# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------

df = standardize_columns(df)


# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------

df = trim_string_columns(df)


# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------

df = blank_to_null(df)


# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------

df = standardize_text(df)


# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------

before_duplicates = df.count()

df = remove_duplicates(df)

duplicates_removed = before_duplicates - df.count()


# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------

df = fill_numeric_nulls(df)


# ------------------------------------------------------------------------------
# Step 9 : Convert Duration to Seconds
#
# Example:
# 32:15 -> 1935 Seconds
# ------------------------------------------------------------------------------

df = df.withColumn(
    "Duration_Seconds",
    (
        F.split(F.col("Duration"), ":").getItem(0).cast("int") * 60
    ) +
    (
        F.split(F.col("Duration"), ":").getItem(1).cast("int")
    )
)


# ------------------------------------------------------------------------------
# Step 10 : Validate Team A Score
# ------------------------------------------------------------------------------

team_a_invalid = df.filter(

    F.col("Team_A_Score") !=
    (
        F.col("Team_A_Attacker_Score") +
        F.col("Team_A_Defender_Score") +
        F.col("Team_A_Overtime_Score")
    )

).count()

print(f"Invalid Team A Scores : {team_a_invalid}")


# ------------------------------------------------------------------------------
# Step 11 : Validate Team B Score
# ------------------------------------------------------------------------------

team_b_invalid = df.filter(

    F.col("Team_B_Score") !=
    (
        F.col("Team_B_Attacker_Score") +
        F.col("Team_B_Defender_Score") +
        F.col("Team_B_Overtime_Score")
    )

).count()

print(f"Invalid Team B Scores : {team_b_invalid}")


# ------------------------------------------------------------------------------
# Step 12 : Remove Invalid Score Records
# ------------------------------------------------------------------------------

df = df.filter(

    (F.col("Team_A_Score") ==
        F.col("Team_A_Attacker_Score") +
        F.col("Team_A_Defender_Score") +
        F.col("Team_A_Overtime_Score"))

    &

    (F.col("Team_B_Score") ==
        F.col("Team_B_Attacker_Score") +
        F.col("Team_B_Defender_Score") +
        F.col("Team_B_Overtime_Score"))

)


# ------------------------------------------------------------------------------
# Step 13 : Remove Empty Rows
# ------------------------------------------------------------------------------

df = remove_empty_rows(df)


# ------------------------------------------------------------------------------
# Step 14 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (After Cleaning)")

quality_report(df)

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 15 : Write Silver Dataset
# ------------------------------------------------------------------------------

write_silver(df, dataset_name)


# ------------------------------------------------------------------------------
# Step 16 : Validate Silver Dataset
# ------------------------------------------------------------------------------

validate_silver(dataset_name)


# ------------------------------------------------------------------------------
# Step 17 : ETL Summary
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)

print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Invalid Team A Rows  : {team_a_invalid}")
print(f"Invalid Team B Rows  : {team_b_invalid}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_PATH}/{dataset_name}")
print("Status               : SUCCESS")

print("=" * 80)



Processing Dataset : maps_scores
Rows Read : 1277


Tournament,Stage,Match Type,Match Name,Map,Team A,Team A Score,Team A Attacker Score,Team A Defender Score,Team A Overtime Score,Team B,Team B Score,Team B Attacker Score,Team B Defender Score,Team B Overtime Score,Duration
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,Paper Rex,13,6,7,null,Xi Lai Gaming,9,3,6,null,48:56
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Sunset,Paper Rex,13,8,5,null,Xi Lai Gaming,5,1,4,null,39:04
Valorant Champions 2025,Group Stage,Opening (A),GIANTX vs Sentinels,Corrode,GIANTX,6,6,0,null,Sentinels,13,7,6,null,41:17
Valorant Champions 2025,Group Stage,Opening (A),GIANTX vs Sentinels,Sunset,GIANTX,13,10,3,null,Sentinels,4,2,2,null,38:15
Valorant Champions 2025,Group Stage,Opening (A),GIANTX vs Sentinels,Haven,GIANTX,13,4,9,null,Sentinels,9,1,8,null,51:34
Valorant Champions 2025,Group Stage,Opening (C),NRG vs EDward Gaming,Abyss,Mega Minors,17,2,10,5,EDward Gaming,15,2,10,3,1:14:38
Valorant Champions 2025,Group Stage,Opening (C),NRG vs EDward Gaming,Corrode,Mega Minors,13,6,7,null,EDward Gaming,11,5,6,null,55:36
Valorant Champions 2025,Group Stage,Opening (C),Team Liquid vs DRX,Abyss,Team Liquid,8,6,2,null,DRX,13,7,6,null,47:56
Valorant Champions 2025,Group Stage,Opening (C),Team Liquid vs DRX,Bind,Team Liquid,10,6,4,null,DRX,13,7,6,null,51:28
Valorant Champions 2025,Group Stage,Opening (D),Dragon Ranger Gaming vs T1,Lotus,Dragon Ranger Gaming,13,5,7,1,T1,15,5,7,3,1:04:38



Data Quality Report (Before Cleaning)
Rows    : 1277
Columns : 16


Tournament,Stage,Match Type,Match Name,Map,Team A,Team A Score,Team A Attacker Score,Team A Defender Score,Team A Overtime Score,Team B,Team B Score,Team B Attacker Score,Team B Defender Score,Team B Overtime Score,Duration
0,0,0,0,0,0,0,0,0,1132,0,0,0,0,1132,98


Duplicate Rows Removed : 0
Invalid Team A Scores : 1
Invalid Team B Scores : 1

Data Quality Report (After Cleaning)
Rows    : 1276
Columns : 17


tournament,stage,match_type,match_name,map,team_a,team_a_score,team_a_attacker_score,team_a_defender_score,team_a_overtime_score,team_b,team_b_score,team_b_attacker_score,team_b_defender_score,team_b_overtime_score,duration,Duration_Seconds
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,97,97


tournament,stage,match_type,match_name,map,team_a,team_a_score,team_a_attacker_score,team_a_defender_score,team_a_overtime_score,team_b,team_b_score,team_b_attacker_score,team_b_defender_score,team_b_overtime_score,duration,Duration_Seconds
Valorant Champions 2025,Group Stage,Winner's (d),Team Heretics Vs T1,Lotus,Team Heretics,13,8,5,0,T1,10,6,4,0,51:06,3066
Valorant Champions 2025,Group Stage,Decider (a),Giantx Vs Xi Lai Gaming,Ascent,Giantx,13,5,8,0,Xi Lai Gaming,10,3,7,0,47:59,2879
Valorant Champions 2025,Playoffs,Upper Quarterfinals,Fnatic Vs Drx,Ascent,Fnatic,8,6,2,0,Drx,13,7,6,0,49:01,2941
Valorant Champions 2025,Playoffs,Upper Semifinals,Mibr Vs Nrg,Abyss,Mibr,14,6,6,2,Mega Minors,16,6,6,4,1:10:57,70
Valorant Champions 2025,Playoffs,Lower Round 2,Mibr Vs Drx,Sunset,Mibr,13,3,10,0,Drx,10,1,9,0,47:50,2870
Valorant Champions 2025,Playoffs,Lower Round 2,Mibr Vs Drx,Ascent,Mibr,13,5,7,1,Drx,15,5,7,3,1:06:08,66
Vct 2025: Americas Stage 2,Group Stage,Week 1,Sentinels Vs G2 Esports,Corrode,Sentinels,13,10,3,0,G2 Esports,8,6,2,0,55:23,3323
Vct 2025: Americas Stage 2,Group Stage,Week 5,100 Thieves Vs Leviatán,Haven,100 Thieves,13,3,10,0,Leviatán,10,1,9,0,54:45,3285
Vct 2025: Americas Stage 2,Group Stage,Week 5,G2 Esports Vs Cloud9,Lotus,G2 Esports,13,8,5,0,Cloud9,9,5,4,0,49:26,2966
Vct 2025: Americas Stage 2,Playoffs,Upper Round 1,Leviatán Vs Cloud9,Corrode,Leviatán,10,8,2,0,Cloud9,13,9,4,0,1:08:00,68


Silver Dataset Written Successfully
/Volumes/workspace/default/matrica/silver/vct_2025/matches/maps_scores
Validation Successful
Rows : 1276
Columns : 17


tournament,stage,match_type,match_name,map,team_a,team_a_score,team_a_attacker_score,team_a_defender_score,team_a_overtime_score,team_b,team_b_score,team_b_attacker_score,team_b_defender_score,team_b_overtime_score,duration,Duration_Seconds
Valorant Champions 2025,Group Stage,Winner's (d),Team Heretics Vs T1,Lotus,Team Heretics,13,8,5,0,T1,10,6,4,0,51:06,3066
Valorant Champions 2025,Group Stage,Decider (a),Giantx Vs Xi Lai Gaming,Ascent,Giantx,13,5,8,0,Xi Lai Gaming,10,3,7,0,47:59,2879
Valorant Champions 2025,Playoffs,Upper Quarterfinals,Fnatic Vs Drx,Ascent,Fnatic,8,6,2,0,Drx,13,7,6,0,49:01,2941
Valorant Champions 2025,Playoffs,Upper Semifinals,Mibr Vs Nrg,Abyss,Mibr,14,6,6,2,Mega Minors,16,6,6,4,1:10:57,70
Valorant Champions 2025,Playoffs,Lower Round 2,Mibr Vs Drx,Sunset,Mibr,13,3,10,0,Drx,10,1,9,0,47:50,2870
Valorant Champions 2025,Playoffs,Lower Round 2,Mibr Vs Drx,Ascent,Mibr,13,5,7,1,Drx,15,5,7,3,1:06:08,66
Vct 2025: Americas Stage 2,Group Stage,Week 1,Sentinels Vs G2 Esports,Corrode,Sentinels,13,10,3,0,G2 Esports,8,6,2,0,55:23,3323
Vct 2025: Americas Stage 2,Group Stage,Week 5,100 Thieves Vs Leviatán,Haven,100 Thieves,13,3,10,0,Leviatán,10,1,9,0,54:45,3285
Vct 2025: Americas Stage 2,Group Stage,Week 5,G2 Esports Vs Cloud9,Lotus,G2 Esports,13,8,5,0,Cloud9,9,5,4,0,49:26,2966
Vct 2025: Americas Stage 2,Playoffs,Upper Round 1,Leviatán Vs Cloud9,Corrode,Leviatán,10,8,2,0,Cloud9,13,9,4,0,1:08:00,68



ETL EXECUTION SUMMARY
Dataset              : maps_scores
Rows Read            : 1277
Rows Written         : 1276
Duplicates Removed   : 0
Invalid Team A Rows  : 1
Invalid Team B Rows  : 1
Columns              : 17
Silver Location      : /Volumes/workspace/default/matrica/silver/vct_2025/matches/maps_scores
Status               : SUCCESS


In [0]:
# ==============================================================================
# Dataset : overview
# Layer   : Bronze -> Silver
# Purpose : Clean & Transform Player Overview Statistics
# ==============================================================================

from pyspark.sql import functions as F

print("=" * 80)
print("Processing Dataset : overview")
print("=" * 80)

# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------

dataset_name = "overview"

df = spark.read.parquet(f"{BRONZE_PATH}/{dataset_name}")

rows_before = df.count()

print(f"Rows Read : {rows_before}")

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (Before Cleaning)")

quality_report(df)


# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------

df = standardize_columns(df)


# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------

df = trim_string_columns(df)


# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------

df = blank_to_null(df)


# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------

df = standardize_text(df)


# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------

before_duplicates = df.count()

df = remove_duplicates(df)

duplicates_removed = before_duplicates - df.count()


# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------

df = fill_numeric_nulls(df)


# ------------------------------------------------------------------------------
# Step 9 : Convert Percentage Columns
# ------------------------------------------------------------------------------

df = df.withColumn(
    "headshot_%",
    F.regexp_replace(F.col("headshot_%"), "%", "").cast("double")
)

df = df.withColumn(
    "kill,_assist,_trade,_survive_%",
    F.regexp_replace(
        F.col("kill,_assist,_trade,_survive_%"),
        "%",
        ""
    ).cast("double")
)


# ------------------------------------------------------------------------------
# Step 10 : Recalculate KD Difference
# ------------------------------------------------------------------------------

df = df.withColumn(
    "kills_deaths_kd",
    F.col("kills") - F.col("deaths")
)


# ------------------------------------------------------------------------------
# Step 11 : Recalculate FK Difference
# ------------------------------------------------------------------------------

df = df.withColumn(
    "first_kills_deaths_fkd",
    F.col("first_kills") - F.col("first_deaths")
)


# ------------------------------------------------------------------------------
# Step 12 : Rating Validation
# ------------------------------------------------------------------------------

invalid_rating = df.filter(F.col("rating") < 0).count()

print(f"Invalid Ratings : {invalid_rating}")

df = df.filter(F.col("rating") >= 0)


# ------------------------------------------------------------------------------
# Step 13 : Remove Empty Rows
# ------------------------------------------------------------------------------

df = remove_empty_rows(df)


# ------------------------------------------------------------------------------
# Step 14 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (After Cleaning)")

quality_report(df)

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 15 : Write Silver Dataset
# ------------------------------------------------------------------------------

write_silver(df, dataset_name)


# ------------------------------------------------------------------------------
# Step 16 : Validate Silver Dataset
# ------------------------------------------------------------------------------

validate_silver(dataset_name)


# ------------------------------------------------------------------------------
# Step 17 : ETL Summary
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)

print(f"Dataset                : {dataset_name}")
print(f"Rows Read              : {rows_before}")
print(f"Rows Written           : {df.count()}")
print(f"Duplicates Removed     : {duplicates_removed}")
print(f"Invalid Ratings        : {invalid_rating}")
print(f"Columns                : {len(df.columns)}")
print(f"Silver Location        : {SILVER_PATH}/{dataset_name}")
print("Status                 : SUCCESS")

print("=" * 80)

Processing Dataset : overview
Rows Read : 53226


Tournament,Stage,Match Type,Match Name,Map,Player,Team,Agents,Rating,Average Combat Score,Kills,Deaths,Assists,Kills - Deaths (KD),"Kill, Assist, Trade, Survive %",Average Damage Per Round,Headshot %,First Kills,First Deaths,Kills - Deaths (FKD),Side
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,something,Paper Rex,yoru,1.63,258,21,12,12,9,91%,167,15%,4,0,4,both
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,something,Paper Rex,yoru,1.33,199,7,5,6,2,100%,137,16%,1,0,1,attack
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,something,Paper Rex,yoru,1.88,308,14,7,6,7,83%,192,13%,3,0,3,defend
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,Jinggg,Paper Rex,raze,1.52,315,24,14,7,10,82%,207,18%,5,2,3,both
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,Jinggg,Paper Rex,raze,1.11,239,8,7,2,1,80%,148,13%,3,2,1,attack
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,Jinggg,Paper Rex,raze,1.87,380,16,7,5,9,83%,256,21%,2,0,2,defend
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,f0rsakeN,Paper Rex,brimstone,1.17,217,18,13,13,5,77%,150,36%,2,5,-3,both
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,f0rsakeN,Paper Rex,brimstone,1.99,341,14,4,7,10,100%,218,38%,1,1,0,attack
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,f0rsakeN,Paper Rex,brimstone,0.49,114,4,9,6,-5,58%,94,32%,1,4,-3,defend
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,d4v41,Paper Rex,viper,0.94,176,14,14,5,0,82%,129,21%,1,1,0,both



Data Quality Report (Before Cleaning)
Rows    : 53226
Columns : 21


Tournament,Stage,Match Type,Match Name,Map,Player,Team,Agents,Rating,Average Combat Score,Kills,Deaths,Assists,Kills - Deaths (KD),"Kill, Assist, Trade, Survive %",Average Damage Per Round,Headshot %,First Kills,First Deaths,Kills - Deaths (FKD),Side
0,0,0,0,0,0,0,0,4080,2745,2550,2550,2550,2556,3882,4056,3849,3960,3957,3957,0


Duplicate Rows Removed : 0
Invalid Ratings : 0

Data Quality Report (After Cleaning)
Rows    : 53226
Columns : 23


tournament,stage,match_type,match_name,map,player,team,agents,rating,average_combat_score,kills,deaths,assists,kills___deaths_(kd),"kill,_assist,_trade,_survive_%",average_damage_per_round,headshot_%,first_kills,first_deaths,kills___deaths_(fkd),side,kills_deaths_kd,first_kills_deaths_fkd
0,0,0,0,0,0,0,0,0,0,0,0,0,0,3882,0,3849,0,0,0,0,0,0


tournament,stage,match_type,match_name,map,player,team,agents,rating,average_combat_score,kills,deaths,assists,kills___deaths_(kd),"kill,_assist,_trade,_survive_%",average_damage_per_round,headshot_%,first_kills,first_deaths,kills___deaths_(fkd),side,kills_deaths_kd,first_kills_deaths_fkd
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,All Maps,F0rsaken,Paper Rex,"Brimstone, Omen",1.09,321,22,9,10,13,94.0,190,35.0,3,1,2,Attack,13,2
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,All Maps,Happywei,Xi Lai Gaming,"Cypher, Viper",1.0,165,21,32,13,-11,55.0,122,36.0,3,8,-5,Both,-11,-5
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,All Maps,Flickless,Giantx,"Breach, Viper",0.8,159,32,32,27,0,84.0,106,26.0,1,5,-4,Both,0,-4
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,All Maps,Bang,Sentinels,Omen,1.02,117,14,24,9,-10,61.0,70,38.0,3,6,-3,Attack,-10,-3
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Sunset,Zellsis,Sentinels,Breach,0.53,125,7,15,6,-8,59.0,88,17.0,1,2,-1,Both,-8,-1
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Sunset,Zellsis,Sentinels,Breach,0.22,128,2,5,2,-3,80.0,77,8.0,1,0,1,Defend,-3,1
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Sunset,Bang,Sentinels,Omen,0.05,34,1,10,2,-9,33.0,23,33.0,0,2,-2,Attack,-9,-2
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Haven,Grubinho,Giantx,Omen,0.99,196,17,14,7,3,82.0,119,30.0,1,3,-2,Both,3,-2
Valorant Champions 2025,Group Stage,Opening (c),Nrg Vs Edward Gaming,Abyss,S0m,Mega Minors,Harbor,0.81,167,18,24,10,-6,66.0,110,28.0,2,4,-2,Both,-6,-2
Valorant Champions 2025,Group Stage,Opening (c),Nrg Vs Edward Gaming,All Maps,Ethan,Mega Minors,"Kayo, Omen",0.76,273,26,17,17,9,75.0,190,37.0,2,5,-3,Attack,9,-3


Silver Dataset Written Successfully
/Volumes/workspace/default/matrica/silver/vct_2025/matches/overview
Validation Successful
Rows : 53226
Columns : 23


tournament,stage,match_type,match_name,map,player,team,agents,rating,average_combat_score,kills,deaths,assists,kills___deaths_(kd),"kill,_assist,_trade,_survive_%",average_damage_per_round,headshot_%,first_kills,first_deaths,kills___deaths_(fkd),side,kills_deaths_kd,first_kills_deaths_fkd
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,D4v41,Paper Rex,Viper,0.59,126,5,9,4,-4,67.0,98,38.0,0,1,-1,Defend,-4,-1
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,Patmen,Paper Rex,Fade,0.88,161,12,13,5,-1,68.0,110,34.0,0,2,-2,Both,-1,-2
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,Noman,Xi Lai Gaming,Skye,1.06,249,11,10,0,1,75.0,157,46.0,0,0,0,Attack,1,0
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,All Maps,Something,Paper Rex,"Sova, Yoru",1.59,271,15,8,9,7,94.0,189,28.0,1,1,0,Attack,7,0
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Sunset,F0rsaken,Paper Rex,Omen,1.37,288,8,5,3,3,83.0,144,29.0,2,0,2,Attack,3,2
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Sunset,Noman,Xi Lai Gaming,Neon,1.84,415,9,5,2,4,100.0,248,36.0,1,0,1,Defend,4,1
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Sunset,Viva,Xi Lai Gaming,Sova,0.57,120,5,9,3,-4,75.0,88,27.0,0,0,0,Attack,-4,0
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Corrode,Cloud,Giantx,Fade,0.93,133,5,7,3,-2,50.0,96,22.0,2,1,1,Defend,-2,1
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Corrode,Westside,Giantx,Vyse,0.15,25,0,7,0,-7,null,25,0.0,0,0,0,Attack,-7,0
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Corrode,Zellsis,Sentinels,Vyse,0.96,125,9,8,5,1,84.0,76,16.0,1,0,1,Both,1,1



ETL EXECUTION SUMMARY
Dataset                : overview
Rows Read              : 53226
Rows Written           : 53226
Duplicates Removed     : 0
Invalid Ratings        : 0
Columns                : 23
Silver Location        : /Volumes/workspace/default/matrica/silver/vct_2025/matches/overview
Status                 : SUCCESS


In [0]:
# ==============================================================================
# Dataset : rounds_kills
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate Round Kill Events
# ==============================================================================

from pyspark.sql import functions as F

print("=" * 80)
print("Processing Dataset : rounds_kills")
print("=" * 80)

# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------

dataset_name = "rounds_kills"

df = spark.read.parquet(f"{BRONZE_PATH}/{dataset_name}")

rows_before = df.count()

print(f"Rows Read : {rows_before}")

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (Before Cleaning)")

quality_report(df)


# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------

df = standardize_columns(df)


# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------

df = trim_string_columns(df)


# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------

df = blank_to_null(df)


# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------

df = standardize_text(df)


# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------

before_duplicates = df.count()

df = remove_duplicates(df)

duplicates_removed = before_duplicates - df.count()


# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------

df = fill_numeric_nulls(df)


# ------------------------------------------------------------------------------
# Step 9 : Validate Round Number
#
# Valorant rounds normally range from 1 to 30
# ------------------------------------------------------------------------------

invalid_rounds = df.filter(
    (F.col("Round_Number") < 1) |
    (F.col("Round_Number") > 30)
).count()

print(f"Invalid Round Records : {invalid_rounds}")

df = df.filter(
    (F.col("Round_Number") >= 1) &
    (F.col("Round_Number") <= 30)
)


# ------------------------------------------------------------------------------
# Step 10 : Remove Records with Missing Players
# ------------------------------------------------------------------------------

missing_players = df.filter(
    F.col("Eliminator").isNull() |
    F.col("Eliminated").isNull()
).count()

print(f"Rows Removed (Missing Players) : {missing_players}")

df = df.filter(
    F.col("Eliminator").isNotNull() &
    F.col("Eliminated").isNotNull()
)


# ------------------------------------------------------------------------------
# Step 11 : Remove Completely Empty Rows
# ------------------------------------------------------------------------------

df = remove_empty_rows(df)


# ------------------------------------------------------------------------------
# Step 12 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (After Cleaning)")

quality_report(df)

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 13 : Write Silver Dataset
# ------------------------------------------------------------------------------

write_silver(df, dataset_name)


# ------------------------------------------------------------------------------
# Step 14 : Validate Silver Dataset
# ------------------------------------------------------------------------------

validate_silver(dataset_name)


# ------------------------------------------------------------------------------
# Step 15 : ETL Summary
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)

print(f"Dataset                 : {dataset_name}")
print(f"Rows Read               : {rows_before}")
print(f"Rows Written            : {df.count()}")
print(f"Duplicates Removed      : {duplicates_removed}")
print(f"Invalid Round Records   : {invalid_rounds}")
print(f"Missing Player Records  : {missing_players}")
print(f"Columns                 : {len(df.columns)}")
print(f"Silver Location         : {SILVER_PATH}/{dataset_name}")
print("Status                  : SUCCESS")

print("=" * 80)

Processing Dataset : rounds_kills
Rows Read : 89088


Tournament,Stage,Match Type,Match Name,Map,Round Number,Eliminator Team,Eliminator,Eliminator Agent,Eliminated Team,Eliminated,Eliminated Agent,Kill Type
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,3,Paper Rex,Jinggg,raze,Xi Lai Gaming,coconut,gekko,2k
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,3,Paper Rex,Jinggg,raze,Xi Lai Gaming,NoMan,skye,2k
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,5,Paper Rex,Jinggg,raze,Xi Lai Gaming,Rarga,raze,2k
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,5,Paper Rex,Jinggg,raze,Xi Lai Gaming,NoMan,skye,2k
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,13,Paper Rex,Jinggg,raze,Xi Lai Gaming,Viva,brimstone,2k
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,13,Paper Rex,Jinggg,raze,Xi Lai Gaming,Rarga,raze,2k
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,14,Paper Rex,Jinggg,raze,Xi Lai Gaming,coconut,gekko,2k
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,14,Paper Rex,Jinggg,raze,Xi Lai Gaming,NoMan,skye,2k
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,21,Paper Rex,Jinggg,raze,Xi Lai Gaming,NoMan,skye,2k
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,21,Paper Rex,Jinggg,raze,Xi Lai Gaming,Viva,brimstone,2k



Data Quality Report (Before Cleaning)
Rows    : 89088
Columns : 13


Tournament,Stage,Match Type,Match Name,Map,Round Number,Eliminator Team,Eliminator,Eliminator Agent,Eliminated Team,Eliminated,Eliminated Agent,Kill Type
0,0,0,0,0,0,0,0,0,0,0,0,0


Duplicate Rows Removed : 37
Invalid Round Records : 324
Rows Removed (Missing Players) : 0

Data Quality Report (After Cleaning)
Rows    : 88727
Columns : 13


tournament,stage,match_type,match_name,map,round_number,eliminator_team,eliminator,eliminator_agent,eliminated_team,eliminated,eliminated_agent,kill_type
0,0,0,0,0,0,0,0,0,0,0,0,0


tournament,stage,match_type,match_name,map,round_number,eliminator_team,eliminator,eliminator_agent,eliminated_team,eliminated,eliminated_agent,kill_type
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,21,Paper Rex,Jinggg,Raze,Xi Lai Gaming,Noman,Skye,2k
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,6,Paper Rex,Something,Yoru,Xi Lai Gaming,Viva,Brimstone,2k
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,11,Paper Rex,Something,Yoru,Xi Lai Gaming,Noman,Skye,1v2
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,6,Xi Lai Gaming,Noman,Skye,Paper Rex,Patmen,Fade,2k
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,19,Xi Lai Gaming,Noman,Skye,Paper Rex,Something,Yoru,2k
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Sunset,9,Paper Rex,Jinggg,Raze,Xi Lai Gaming,Viva,Sova,2k
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Sunset,8,Paper Rex,F0rsaken,Omen,Xi Lai Gaming,Happywei,Cypher,4k
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Sunset,10,Paper Rex,Something,Sova,Xi Lai Gaming,Noman,Neon,2k
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Sunset,14,Paper Rex,Something,Sova,Xi Lai Gaming,Rarga,Yoru,2k
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Sunset,15,Xi Lai Gaming,Noman,Neon,Paper Rex,Jinggg,Raze,2k


Silver Dataset Written Successfully
/Volumes/workspace/default/matrica/silver/vct_2025/matches/rounds_kills
Validation Successful
Rows : 88727
Columns : 13


tournament,stage,match_type,match_name,map,round_number,eliminator_team,eliminator,eliminator_agent,eliminated_team,eliminated,eliminated_agent,kill_type
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,13,Paper Rex,Jinggg,Raze,Xi Lai Gaming,Viva,Brimstone,2k
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,11,Paper Rex,Something,Yoru,Xi Lai Gaming,Happywei,Viper,4k
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,4,Xi Lai Gaming,Coconut,Gekko,Paper Rex,Something,Yoru,2k
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,13,Xi Lai Gaming,Coconut,Gekko,Paper Rex,F0rsaken,Brimstone,2k
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,15,Xi Lai Gaming,Rarga,Raze,Paper Rex,Patmen,Fade,2k
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Sunset,6,Paper Rex,Jinggg,Raze,Xi Lai Gaming,Noman,Neon,2k
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Sunset,1,Paper Rex,F0rsaken,Omen,Xi Lai Gaming,Coconut,Omen,2k
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Sunset,8,Paper Rex,F0rsaken,Omen,Xi Lai Gaming,Coconut,Omen,4k
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Sunset,14,Paper Rex,D4v41,Sage,Xi Lai Gaming,Noman,Neon,2k
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Sunset,16,Paper Rex,D4v41,Sage,Xi Lai Gaming,Noman,Neon,1v1



ETL EXECUTION SUMMARY
Dataset                 : rounds_kills
Rows Read               : 89088
Rows Written            : 88727
Duplicates Removed      : 37
Invalid Round Records   : 324
Missing Player Records  : 0
Columns                 : 13
Silver Location         : /Volumes/workspace/default/matrica/silver/vct_2025/matches/rounds_kills
Status                  : SUCCESS


In [0]:
# ==============================================================================
# Dataset : scores
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate Match Scores
# ==============================================================================

from pyspark.sql import functions as F

print("=" * 80)
print("Processing Dataset : scores")
print("=" * 80)

# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------

dataset_name = "scores"

df = spark.read.parquet(f"{BRONZE_PATH}/{dataset_name}")

rows_before = df.count()

print(f"Rows Read : {rows_before}")

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (Before Cleaning)")

quality_report(df)


# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------

df = standardize_columns(df)


# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------

df = trim_string_columns(df)


# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------

df = blank_to_null(df)


# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------

df = standardize_text(df)


# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------

before_duplicates = df.count()

df = remove_duplicates(df)

duplicates_removed = before_duplicates - df.count()


# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------

df = fill_numeric_nulls(df)


# ------------------------------------------------------------------------------
# Step 9 : Remove Invalid Scores
# ------------------------------------------------------------------------------

invalid_scores = df.filter(
    (F.col("Team_A_Score") < 0) |
    (F.col("Team_B_Score") < 0)
).count()

print(f"Invalid Score Records : {invalid_scores}")

df = df.filter(
    (F.col("Team_A_Score") >= 0) &
    (F.col("Team_B_Score") >= 0)
)


# ------------------------------------------------------------------------------
# Step 10 : Identify Winning Team
# ------------------------------------------------------------------------------

df = df.withColumn(
    "Winner",

    F.when(
        F.col("Team_A_Score") > F.col("Team_B_Score"),
        F.col("Team_A")
    )

    .when(
        F.col("Team_B_Score") > F.col("Team_A_Score"),
        F.col("Team_B")
    )

    .otherwise(F.lit("Draw"))
)


# ------------------------------------------------------------------------------
# Step 11 : Validate Match Result
# ------------------------------------------------------------------------------

df = df.withColumn(

    "Result_Validation",

    F.when(
        F.lower(F.col("Match_Result")) == F.lower(F.col("Winner")),
        "Valid"
    )

    .otherwise("Mismatch")

)

validation_errors = df.filter(
    F.col("Result_Validation") == "Mismatch"
).count()

print(f"Result Validation Errors : {validation_errors}")


# ------------------------------------------------------------------------------
# Step 12 : Remove Empty Rows
# ------------------------------------------------------------------------------

df = remove_empty_rows(df)


# ------------------------------------------------------------------------------
# Step 13 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (After Cleaning)")

quality_report(df)

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 14 : Write Silver Dataset
# ------------------------------------------------------------------------------

write_silver(df, dataset_name)


# ------------------------------------------------------------------------------
# Step 15 : Validate Silver Dataset
# ------------------------------------------------------------------------------

validate_silver(dataset_name)


# ------------------------------------------------------------------------------
# Step 16 : ETL Summary
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)

print(f"Dataset                 : {dataset_name}")
print(f"Rows Read               : {rows_before}")
print(f"Rows Written            : {df.count()}")
print(f"Duplicates Removed      : {duplicates_removed}")
print(f"Invalid Scores          : {invalid_scores}")
print(f"Result Mismatches       : {validation_errors}")
print(f"Columns                 : {len(df.columns)}")
print(f"Silver Location         : {SILVER_PATH}/{dataset_name}")
print("Status                  : SUCCESS")

print("=" * 80)

Processing Dataset : scores
Rows Read : 503


Tournament,Stage,Match Type,Match Name,Team A,Team B,Team A Score,Team B Score,Match Result
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Paper Rex,Xi Lai Gaming,2,0,Paper Rex won
Valorant Champions 2025,Group Stage,Opening (A),GIANTX vs Sentinels,GIANTX,Sentinels,2,1,GIANTX won
Valorant Champions 2025,Group Stage,Opening (C),NRG vs EDward Gaming,Mega Minors,EDward Gaming,2,0,NRG won
Valorant Champions 2025,Group Stage,Opening (C),Team Liquid vs DRX,Team Liquid,DRX,0,2,DRX won
Valorant Champions 2025,Group Stage,Opening (D),Dragon Ranger Gaming vs T1,Dragon Ranger Gaming,T1,0,2,T1 won
Valorant Champions 2025,Group Stage,Opening (D),G2 Esports vs Team Heretics,G2 Esports,Team Heretics,0,2,Team Heretics won
Valorant Champions 2025,Group Stage,Opening (B),Bilibili Gaming vs MIBR,Bilibili Gaming,MIBR,0,2,MIBR won
Valorant Champions 2025,Group Stage,Opening (B),Rex Regum Qeon vs FNATIC,Rex Regum Qeon,FNATIC,0,2,FNATIC won
Valorant Champions 2025,Group Stage,Winner's (A),Paper Rex vs GIANTX,Paper Rex,GIANTX,2,1,Paper Rex won
Valorant Champions 2025,Group Stage,Winner's (C),DRX vs NRG,DRX,Mega Minors,1,2,NRG won



Data Quality Report (Before Cleaning)
Rows    : 503
Columns : 9


Tournament,Stage,Match Type,Match Name,Team A,Team B,Team A Score,Team B Score,Match Result
0,0,0,0,0,0,0,0,0


Duplicate Rows Removed : 0
Invalid Score Records : 0
Result Validation Errors : 503

Data Quality Report (After Cleaning)
Rows    : 503
Columns : 11


tournament,stage,match_type,match_name,team_a,team_b,team_a_score,team_b_score,match_result,Winner,Result_Validation
0,0,0,0,0,0,0,0,0,0,0


tournament,stage,match_type,match_name,team_a,team_b,team_a_score,team_b_score,match_result,Winner,Result_Validation
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Giantx,Sentinels,2,1,Giantx Won,Giantx,Mismatch
Valorant Champions 2025,Group Stage,Elimination (a),Xi Lai Gaming Vs Sentinels,Xi Lai Gaming,Sentinels,2,1,Xi Lai Gaming Won,Xi Lai Gaming,Mismatch
Vct 2025: Americas Stage 2,Playoffs,Upper Round 1,Leviatán Vs Cloud9,Leviatán,Cloud9,1,2,Cloud9 Won,Cloud9,Mismatch
Vct 2025: Americas Stage 2,Playoffs,Upper Semifinals,Nrg Vs G2 Esports,Mega Minors,G2 Esports,1,2,G2 Esports Won,G2 Esports,Mismatch
Vct 2025: Americas Stage 2,Playoffs,Lower Round 1,100 Thieves Vs Evil Geniuses,100 Thieves,Evil Geniuses,2,0,100 Thieves Won,100 Thieves,Mismatch
Vct 2025: Emea Stage 2,Group Stage,Week 1,Natus Vincere Vs Apeks,Natus Vincere,Apeks,2,1,Natus Vincere Won,Natus Vincere,Mismatch
Vct 2025: Emea Stage 2,Group Stage,Week 3,Team Liquid Vs Apeks,Team Liquid,Apeks,2,1,Team Liquid Won,Team Liquid,Mismatch
Vct 2025: Emea Stage 2,Playoffs,Upper Final,Bbl Esports Vs Team Liquid,Bbl Esports,Team Liquid,1,2,Team Liquid Won,Team Liquid,Mismatch
Vct 2025: Pacific Stage 2,Group Stage,Week 2,Boom Esports Vs Paper Rex,Boom Esports,Paper Rex,0,2,Paper Rex Won,Paper Rex,Mismatch
Vct 2025: China Stage 2,Group Stage,Week 2,Tyloo Vs Titan Esports Club,Tyloo,Titan Esports Club,2,0,Tyloo Won,Tyloo,Mismatch


Silver Dataset Written Successfully
/Volumes/workspace/default/matrica/silver/vct_2025/matches/scores
Validation Successful
Rows : 503
Columns : 11


tournament,stage,match_type,match_name,team_a,team_b,team_a_score,team_b_score,match_result,Winner,Result_Validation
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Giantx,Sentinels,2,1,Giantx Won,Giantx,Mismatch
Valorant Champions 2025,Group Stage,Elimination (a),Xi Lai Gaming Vs Sentinels,Xi Lai Gaming,Sentinels,2,1,Xi Lai Gaming Won,Xi Lai Gaming,Mismatch
Vct 2025: Americas Stage 2,Playoffs,Upper Round 1,Leviatán Vs Cloud9,Leviatán,Cloud9,1,2,Cloud9 Won,Cloud9,Mismatch
Vct 2025: Americas Stage 2,Playoffs,Upper Semifinals,Nrg Vs G2 Esports,Mega Minors,G2 Esports,1,2,G2 Esports Won,G2 Esports,Mismatch
Vct 2025: Americas Stage 2,Playoffs,Lower Round 1,100 Thieves Vs Evil Geniuses,100 Thieves,Evil Geniuses,2,0,100 Thieves Won,100 Thieves,Mismatch
Vct 2025: Emea Stage 2,Group Stage,Week 1,Natus Vincere Vs Apeks,Natus Vincere,Apeks,2,1,Natus Vincere Won,Natus Vincere,Mismatch
Vct 2025: Emea Stage 2,Group Stage,Week 3,Team Liquid Vs Apeks,Team Liquid,Apeks,2,1,Team Liquid Won,Team Liquid,Mismatch
Vct 2025: Emea Stage 2,Playoffs,Upper Final,Bbl Esports Vs Team Liquid,Bbl Esports,Team Liquid,1,2,Team Liquid Won,Team Liquid,Mismatch
Vct 2025: Pacific Stage 2,Group Stage,Week 2,Boom Esports Vs Paper Rex,Boom Esports,Paper Rex,0,2,Paper Rex Won,Paper Rex,Mismatch
Vct 2025: China Stage 2,Group Stage,Week 2,Tyloo Vs Titan Esports Club,Tyloo,Titan Esports Club,2,0,Tyloo Won,Tyloo,Mismatch



ETL EXECUTION SUMMARY
Dataset                 : scores
Rows Read               : 503
Rows Written            : 503
Duplicates Removed      : 0
Invalid Scores          : 0
Result Mismatches       : 503
Columns                 : 11
Silver Location         : /Volumes/workspace/default/matrica/silver/vct_2025/matches/scores
Status                  : SUCCESS


In [0]:
# ==============================================================================
# Dataset : team_mapping
# Layer   : Bronze -> Silver
# Purpose : Clean & Standardize Team Mapping Lookup Table
# ==============================================================================

from pyspark.sql import functions as F

print("=" * 80)
print("Processing Dataset : team_mapping")
print("=" * 80)

# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------

dataset_name = "team_mapping"

df = spark.read.parquet(f"{BRONZE_PATH}/{dataset_name}")

rows_before = df.count()

print(f"Rows Read : {rows_before}")

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (Before Cleaning)")

quality_report(df)


# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------

df = standardize_columns(df)


# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------

df = trim_string_columns(df)


# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------

df = blank_to_null(df)


# ------------------------------------------------------------------------------
# Step 6 : Standardize Text Values
# ------------------------------------------------------------------------------

df = standardize_text(df)


# ------------------------------------------------------------------------------
# Step 7 : Convert Abbreviations to Uppercase
#
# Example:
# Sen -> SEN
# g2  -> G2
# ------------------------------------------------------------------------------

df = df.withColumn(
    "Abbreviated",
    F.upper(F.col("Abbreviated"))
)


# ------------------------------------------------------------------------------
# Step 8 : Remove Duplicate Abbreviations
# ------------------------------------------------------------------------------

before_duplicates = df.count()

df = df.dropDuplicates(["Abbreviated"])

duplicates_removed = before_duplicates - df.count()


# ------------------------------------------------------------------------------
# Step 9 : Remove Records with Missing Values
# ------------------------------------------------------------------------------

missing_records = df.filter(
    F.col("Abbreviated").isNull() |
    F.col("Full_Name").isNull()
).count()

print(f"Missing Records : {missing_records}")

df = df.filter(
    F.col("Abbreviated").isNotNull() &
    F.col("Full_Name").isNotNull()
)


# ------------------------------------------------------------------------------
# Step 10 : Remove Completely Empty Rows
# ------------------------------------------------------------------------------

df = remove_empty_rows(df)


# ------------------------------------------------------------------------------
# Step 11 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (After Cleaning)")

quality_report(df)

display(df)


# ------------------------------------------------------------------------------
# Step 12 : Write Silver Dataset
# ------------------------------------------------------------------------------

write_silver(df, dataset_name)


# ------------------------------------------------------------------------------
# Step 13 : Validate Silver Dataset
# ------------------------------------------------------------------------------

validate_silver(dataset_name)


# ------------------------------------------------------------------------------
# Step 14 : ETL Summary
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)

print(f"Dataset                 : {dataset_name}")
print(f"Rows Read               : {rows_before}")
print(f"Rows Written            : {df.count()}")
print(f"Duplicates Removed      : {duplicates_removed}")
print(f"Missing Records Removed : {missing_records}")
print(f"Columns                 : {len(df.columns)}")
print(f"Silver Location         : {SILVER_PATH}/{dataset_name}")
print("Status                  : SUCCESS")

print("=" * 80)

Processing Dataset : team_mapping
Rows Read : 58


Abbreviated,Full Name
PRX,Paper Rex
XLG,Xi Lai Gaming
GX,GIANTX
SEN,Sentinels
NRG,NRG
EDG,EDward Gaming
TL,Team Liquid
DRX,DRX
DRG,Dragon Ranger Gaming
T1,T1



Data Quality Report (Before Cleaning)
Rows    : 58
Columns : 2


Abbreviated,Full Name
0,0


Missing Records : 0

Data Quality Report (After Cleaning)
Rows    : 58
Columns : 2


Abbreviated,full_name
0,0


Abbreviated,full_name
100T,100 Thieves
C9,Cloud9
KRÜ,Krü Esports
VIT,Team Vitality
TYL,Tyloo
PRX,Paper Rex
FNC,Fnatic
2G,2game Esports
NAVI,Natus Vincere
GOA,Glory Once Again


Silver Dataset Written Successfully
/Volumes/workspace/default/matrica/silver/vct_2025/matches/team_mapping
Validation Successful
Rows : 58
Columns : 2


Abbreviated,full_name
100T,100 Thieves
C9,Cloud9
KRÜ,Krü Esports
VIT,Team Vitality
TYL,Tyloo
PRX,Paper Rex
FNC,Fnatic
2G,2game Esports
NAVI,Natus Vincere
GOA,Glory Once Again



ETL EXECUTION SUMMARY
Dataset                 : team_mapping
Rows Read               : 58
Rows Written            : 58
Duplicates Removed      : 0
Missing Records Removed : 0
Columns                 : 2
Silver Location         : /Volumes/workspace/default/matrica/silver/vct_2025/matches/team_mapping
Status                  : SUCCESS


In [0]:
# ==============================================================================
# Dataset : win_loss_methods_count
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate Win/Loss Method Statistics
# ==============================================================================

from pyspark.sql import functions as F

print("=" * 80)
print("Processing Dataset : win_loss_methods_count")
print("=" * 80)

# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------

dataset_name = "win_loss_methods_count"

df = spark.read.parquet(f"{BRONZE_PATH}/{dataset_name}")

rows_before = df.count()

print(f"Rows Read : {rows_before}")

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (Before Cleaning)")

quality_report(df)


# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------

df = standardize_columns(df)


# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------

df = trim_string_columns(df)


# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------

df = blank_to_null(df)


# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------

df = standardize_text(df)


# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------

before_duplicates = df.count()

df = remove_duplicates(df)

duplicates_removed = before_duplicates - df.count()


# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------

df = fill_numeric_nulls(df)


# ------------------------------------------------------------------------------
# Step 9 : Validate Numeric Columns
# ------------------------------------------------------------------------------

numeric_columns = [
    "elimination",
    "detonated",
    "defused",
    "time_expiry_(no_plant)",
    "eliminated",
    "defused_failed",
    "detonation_denied",
    "time_expiry_(failed_to_plant)"
]

invalid_condition = None

for column in numeric_columns:

    condition = F.col(column) < 0

    if invalid_condition is None:
        invalid_condition = condition
    else:
        invalid_condition = invalid_condition | condition

invalid_rows = df.filter(invalid_condition).count()

print(f"Invalid Numeric Records : {invalid_rows}")

df = df.filter(~invalid_condition)


# ------------------------------------------------------------------------------
# Step 10 : Create Total_Rounds Column
# ------------------------------------------------------------------------------

df = df.withColumn(

    "total_rounds",

    F.col("elimination") +
    F.col("detonated") +
    F.col("defused") +
    F.col("time_expiry_(no_plant)") +
    F.col("eliminated") +
    F.col("defused_failed") +
    F.col("detonation_denied") +
    F.col("time_expiry_(failed_to_plant)")

)


# ------------------------------------------------------------------------------
# Step 11 : Remove Empty Rows
# ------------------------------------------------------------------------------

df = remove_empty_rows(df)


# ------------------------------------------------------------------------------
# Step 12 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (After Cleaning)")

quality_report(df)

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 13 : Write Silver Dataset
# ------------------------------------------------------------------------------

write_silver(df, dataset_name)


# ------------------------------------------------------------------------------
# Step 14 : Validate Silver Dataset
# ------------------------------------------------------------------------------

validate_silver(dataset_name)


# ------------------------------------------------------------------------------
# Step 15 : ETL Summary
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)

print(f"Dataset                 : {dataset_name}")
print(f"Rows Read               : {rows_before}")
print(f"Rows Written            : {df.count()}")
print(f"Duplicates Removed      : {duplicates_removed}")
print(f"Invalid Numeric Rows    : {invalid_rows}")
print(f"Columns                 : {len(df.columns)}")
print(f"Silver Location         : {SILVER_PATH}/{dataset_name}")
print("Status                  : SUCCESS")

print("=" * 80)

Processing Dataset : win_loss_methods_count
Rows Read : 2554


Tournament,Stage,Match Type,Match Name,Map,Team,Elimination,Detonated,Defused,Time Expiry (No Plant),Eliminated,Defused Failed,Detonation Denied,Time Expiry (Failed to Plant)
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,Paper Rex,8,1,3,1,3,3,2,1
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,Xi Lai Gaming,3,3,2,1,8,1,3,1
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Sunset,Paper Rex,9,0,4,0,3,2,0,0
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Sunset,Xi Lai Gaming,3,2,0,0,9,0,4,0
Valorant Champions 2025,Group Stage,Opening (A),GIANTX vs Sentinels,Corrode,GIANTX,4,0,1,1,8,3,2,0
Valorant Champions 2025,Group Stage,Opening (A),GIANTX vs Sentinels,Corrode,Sentinels,8,3,2,0,4,0,1,1
Valorant Champions 2025,Group Stage,Opening (A),GIANTX vs Sentinels,Sunset,GIANTX,10,0,2,1,3,0,1,0
Valorant Champions 2025,Group Stage,Opening (A),GIANTX vs Sentinels,Sunset,Sentinels,3,0,1,0,10,0,2,1
Valorant Champions 2025,Group Stage,Opening (A),GIANTX vs Sentinels,Haven,GIANTX,11,1,1,0,6,2,1,0
Valorant Champions 2025,Group Stage,Opening (A),GIANTX vs Sentinels,Haven,Sentinels,6,2,1,0,11,1,1,0



Data Quality Report (Before Cleaning)
Rows    : 2554
Columns : 14


Tournament,Stage,Match Type,Match Name,Map,Team,Elimination,Detonated,Defused,Time Expiry (No Plant),Eliminated,Defused Failed,Detonation Denied,Time Expiry (Failed to Plant)
0,0,0,0,0,0,0,0,0,0,0,0,0,0


Duplicate Rows Removed : 0
Invalid Numeric Records : 0

Data Quality Report (After Cleaning)
Rows    : 2554
Columns : 15


tournament,stage,match_type,match_name,map,team,elimination,detonated,defused,time_expiry_(no_plant),eliminated,defused_failed,detonation_denied,time_expiry_(failed_to_plant),total_rounds
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


tournament,stage,match_type,match_name,map,team,elimination,detonated,defused,time_expiry_(no_plant),eliminated,defused_failed,detonation_denied,time_expiry_(failed_to_plant),total_rounds
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,Xi Lai Gaming,3,3,2,1,8,1,3,1,22
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Haven,Sentinels,6,2,1,0,11,1,1,0,22
Valorant Champions 2025,Group Stage,Opening (d),Dragon Ranger Gaming Vs T1,Sunset,T1,13,1,2,0,9,0,5,0,30
Valorant Champions 2025,Group Stage,Opening (d),G2 Esports Vs Team Heretics,Lotus,G2 Esports,6,0,4,0,6,1,5,1,23
Valorant Champions 2025,Group Stage,Winner's (a),Paper Rex Vs Giantx,Ascent,Paper Rex,8,0,2,1,10,1,2,0,24
Valorant Champions 2025,Group Stage,Winner's (c),Drx Vs Nrg,Lotus,Mega Minors,7,2,4,0,8,1,2,0,24
Valorant Champions 2025,Group Stage,Winner's (d),Team Heretics Vs T1,Sunset,T1,8,2,0,1,8,2,2,1,24
Valorant Champions 2025,Group Stage,Elimination (c),Team Liquid Vs Edward Gaming,Bind,Team Liquid,11,2,2,0,9,1,3,0,28
Valorant Champions 2025,Group Stage,Elimination (b),Bilibili Gaming Vs Rex Regum Qeon,Sunset,Bilibili Gaming,9,2,2,0,4,1,2,0,20
Valorant Champions 2025,Group Stage,Decider (a),Giantx Vs Xi Lai Gaming,Bind,Giantx,8,3,3,0,9,0,3,0,26


Silver Dataset Written Successfully
/Volumes/workspace/default/matrica/silver/vct_2025/matches/win_loss_methods_count
Validation Successful
Rows : 2554
Columns : 15


tournament,stage,match_type,match_name,map,team,elimination,detonated,defused,time_expiry_(no_plant),eliminated,defused_failed,detonation_denied,time_expiry_(failed_to_plant),total_rounds
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,Xi Lai Gaming,3,3,2,1,8,1,3,1,22
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Haven,Sentinels,6,2,1,0,11,1,1,0,22
Valorant Champions 2025,Group Stage,Opening (d),Dragon Ranger Gaming Vs T1,Sunset,T1,13,1,2,0,9,0,5,0,30
Valorant Champions 2025,Group Stage,Opening (d),G2 Esports Vs Team Heretics,Lotus,G2 Esports,6,0,4,0,6,1,5,1,23
Valorant Champions 2025,Group Stage,Winner's (a),Paper Rex Vs Giantx,Ascent,Paper Rex,8,0,2,1,10,1,2,0,24
Valorant Champions 2025,Group Stage,Winner's (c),Drx Vs Nrg,Lotus,Mega Minors,7,2,4,0,8,1,2,0,24
Valorant Champions 2025,Group Stage,Winner's (d),Team Heretics Vs T1,Sunset,T1,8,2,0,1,8,2,2,1,24
Valorant Champions 2025,Group Stage,Elimination (c),Team Liquid Vs Edward Gaming,Bind,Team Liquid,11,2,2,0,9,1,3,0,28
Valorant Champions 2025,Group Stage,Elimination (b),Bilibili Gaming Vs Rex Regum Qeon,Sunset,Bilibili Gaming,9,2,2,0,4,1,2,0,20
Valorant Champions 2025,Group Stage,Decider (a),Giantx Vs Xi Lai Gaming,Bind,Giantx,8,3,3,0,9,0,3,0,26



ETL EXECUTION SUMMARY
Dataset                 : win_loss_methods_count
Rows Read               : 2554
Rows Written            : 2554
Duplicates Removed      : 0
Invalid Numeric Rows    : 0
Columns                 : 15
Silver Location         : /Volumes/workspace/default/matrica/silver/vct_2025/matches/win_loss_methods_count
Status                  : SUCCESS


In [0]:
# ==============================================================================
# Dataset : win_loss_methods_round_number
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate Round-wise Win/Loss Methods
# ==============================================================================

from pyspark.sql import functions as F

print("=" * 80)
print("Processing Dataset : win_loss_methods_round_number")
print("=" * 80)

# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------

dataset_name = "win_loss_methods_round_number"

df = spark.read.parquet(f"{BRONZE_PATH}/{dataset_name}")

rows_before = df.count()

print(f"Rows Read : {rows_before}")

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (Before Cleaning)")

quality_report(df)


# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------

df = standardize_columns(df)


# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------

df = trim_string_columns(df)


# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------

df = blank_to_null(df)


# ------------------------------------------------------------------------------
# Step 6 : Standardize Text Values
# ------------------------------------------------------------------------------

df = standardize_text(df)


# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------

before_duplicates = df.count()

df = remove_duplicates(df)

duplicates_removed = before_duplicates - df.count()


# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------

df = fill_numeric_nulls(df)


# ------------------------------------------------------------------------------
# Step 9 : Validate Round Number
# ------------------------------------------------------------------------------

invalid_rounds = df.filter(F.col("Round_Number") <= 0).count()

print(f"Invalid Round Numbers : {invalid_rounds}")

df = df.filter(F.col("Round_Number") > 0)


# ------------------------------------------------------------------------------
# Step 10 : Standardize Method Values
#
# Example:
# "time expiry(no plant)"
# ->
# "Time Expiry (No Plant)"
# ------------------------------------------------------------------------------

df = df.withColumn(
    "Method",
    F.initcap(F.trim(F.col("Method")))
)


# ------------------------------------------------------------------------------
# Step 11 : Validate Outcome
# ------------------------------------------------------------------------------

valid_outcomes = ["Win", "Loss"]

invalid_outcomes = df.filter(
    ~F.col("Outcome").isin(valid_outcomes)
).count()

print(f"Invalid Outcomes : {invalid_outcomes}")

df = df.filter(
    F.col("Outcome").isin(valid_outcomes)
)


# ------------------------------------------------------------------------------
# Step 12 : Create Round Identifier
# ------------------------------------------------------------------------------

df = df.withColumn(
    "Round_ID",
    F.concat_ws(
        "_",
        F.col("Tournament"),
        F.col("Match_Name"),
        F.col("Map"),
        F.col("Round_Number")
    )
)


# ------------------------------------------------------------------------------
# Step 13 : Remove Completely Empty Rows
# ------------------------------------------------------------------------------

df = remove_empty_rows(df)


# ------------------------------------------------------------------------------
# Step 14 : Data Quality Report
# ------------------------------------------------------------------------------

print("\nData Quality Report (After Cleaning)")

quality_report(df)

display(df.limit(10))


# ------------------------------------------------------------------------------
# Step 15 : Write Silver Dataset
# ------------------------------------------------------------------------------

write_silver(df, dataset_name)


# ------------------------------------------------------------------------------
# Step 16 : Validate Silver Dataset
# ------------------------------------------------------------------------------

validate_silver(dataset_name)


# ------------------------------------------------------------------------------
# Step 17 : ETL Summary
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)

print(f"Dataset                 : {dataset_name}")
print(f"Rows Read               : {rows_before}")
print(f"Rows Written            : {df.count()}")
print(f"Duplicates Removed      : {duplicates_removed}")
print(f"Invalid Round Numbers   : {invalid_rounds}")
print(f"Invalid Outcomes        : {invalid_outcomes}")
print(f"Columns                 : {len(df.columns)}")
print(f"Silver Location         : {SILVER_PATH}/{dataset_name}")
print("Status                  : SUCCESS")

print("=" * 80)

Processing Dataset : win_loss_methods_round_number
Rows Read : 53950


Tournament,Stage,Match Type,Match Name,Map,Round Number,Team,Method,Outcome
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,1,Xi Lai Gaming,Elimination,Win
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,1,Paper Rex,Eliminated,Loss
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,2,Xi Lai Gaming,Detonated,Win
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,2,Paper Rex,Failed Defused,Loss
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,3,Paper Rex,Defused,Win
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,3,Xi Lai Gaming,Detonated Denied,Loss
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,4,Xi Lai Gaming,Elimination,Win
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,4,Paper Rex,Eliminated,Loss
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,5,Paper Rex,Elimination,Win
Valorant Champions 2025,Group Stage,Opening (A),Paper Rex vs Xi Lai Gaming,Bind,5,Xi Lai Gaming,Eliminated,Loss



Data Quality Report (Before Cleaning)
Rows    : 53950
Columns : 9


Tournament,Stage,Match Type,Match Name,Map,Round Number,Team,Method,Outcome
0,0,0,0,0,0,0,0,0


Duplicate Rows Removed : 0
Invalid Round Numbers : 0
Invalid Outcomes : 0

Data Quality Report (After Cleaning)
Rows    : 53950
Columns : 10


tournament,stage,match_type,match_name,map,round_number,team,Method,outcome,Round_ID
0,0,0,0,0,0,0,0,0,0


tournament,stage,match_type,match_name,map,round_number,team,Method,outcome,Round_ID
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,14,Paper Rex,Elimination,Win,Valorant Champions 2025_Paper Rex Vs Xi Lai Gaming_Bind_14
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Corrode,5,Sentinels,Detonated,Win,Valorant Champions 2025_Giantx Vs Sentinels_Corrode_5
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Sunset,1,Sentinels,Detonated Denied,Loss,Valorant Champions 2025_Giantx Vs Sentinels_Sunset_1
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Sunset,9,Giantx,Elimination,Win,Valorant Champions 2025_Giantx Vs Sentinels_Sunset_9
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Sunset,9,Sentinels,Eliminated,Loss,Valorant Champions 2025_Giantx Vs Sentinels_Sunset_9
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Sunset,16,Sentinels,Eliminated,Loss,Valorant Champions 2025_Giantx Vs Sentinels_Sunset_16
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Haven,4,Giantx,Failed Defused,Loss,Valorant Champions 2025_Giantx Vs Sentinels_Haven_4
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Haven,18,Sentinels,Defused,Win,Valorant Champions 2025_Giantx Vs Sentinels_Haven_18
Valorant Champions 2025,Group Stage,Opening (c),Nrg Vs Edward Gaming,Abyss,9,Edward Gaming,Elimination,Win,Valorant Champions 2025_Nrg Vs Edward Gaming_Abyss_9
Valorant Champions 2025,Group Stage,Opening (c),Nrg Vs Edward Gaming,Abyss,18,Mega Minors,Detonated,Win,Valorant Champions 2025_Nrg Vs Edward Gaming_Abyss_18


Silver Dataset Written Successfully
/Volumes/workspace/default/matrica/silver/vct_2025/matches/win_loss_methods_round_number
Validation Successful
Rows : 53950
Columns : 10


tournament,stage,match_type,match_name,map,round_number,team,Method,outcome,Round_ID
Valorant Champions 2025,Group Stage,Opening (a),Paper Rex Vs Xi Lai Gaming,Bind,14,Paper Rex,Elimination,Win,Valorant Champions 2025_Paper Rex Vs Xi Lai Gaming_Bind_14
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Corrode,5,Sentinels,Detonated,Win,Valorant Champions 2025_Giantx Vs Sentinels_Corrode_5
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Sunset,1,Sentinels,Detonated Denied,Loss,Valorant Champions 2025_Giantx Vs Sentinels_Sunset_1
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Sunset,9,Giantx,Elimination,Win,Valorant Champions 2025_Giantx Vs Sentinels_Sunset_9
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Sunset,9,Sentinels,Eliminated,Loss,Valorant Champions 2025_Giantx Vs Sentinels_Sunset_9
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Sunset,16,Sentinels,Eliminated,Loss,Valorant Champions 2025_Giantx Vs Sentinels_Sunset_16
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Haven,4,Giantx,Failed Defused,Loss,Valorant Champions 2025_Giantx Vs Sentinels_Haven_4
Valorant Champions 2025,Group Stage,Opening (a),Giantx Vs Sentinels,Haven,18,Sentinels,Defused,Win,Valorant Champions 2025_Giantx Vs Sentinels_Haven_18
Valorant Champions 2025,Group Stage,Opening (c),Nrg Vs Edward Gaming,Abyss,9,Edward Gaming,Elimination,Win,Valorant Champions 2025_Nrg Vs Edward Gaming_Abyss_9
Valorant Champions 2025,Group Stage,Opening (c),Nrg Vs Edward Gaming,Abyss,18,Mega Minors,Detonated,Win,Valorant Champions 2025_Nrg Vs Edward Gaming_Abyss_18



ETL EXECUTION SUMMARY
Dataset                 : win_loss_methods_round_number
Rows Read               : 53950
Rows Written            : 53950
Duplicates Removed      : 0
Invalid Round Numbers   : 0
Invalid Outcomes        : 0
Columns                 : 10
Silver Location         : /Volumes/workspace/default/matrica/silver/vct_2025/matches/win_loss_methods_round_number
Status                  : SUCCESS
